# Pick Face v5.7 — 20 seconds, automatic video orientation
Generate a talking video with separate speech audio and choose the character by Face ID.

Use a **fresh Colab runtime** with an **A100 GPU**. Run steps 1–6 in order.
In **step 7**, upload video and speech audio in the Files sidebar, use **Copy path**,
paste both paths, choose a resolution, and click **Use files**.
In **step 8**, Detect faces → Face ID → Preview selected face → Generate video.
Use **step 9** to view/download results and diagnostics. No Google Drive or public tunnel is needed.

Processes the first **up to 20 seconds at 25 FPS** (500 input frames).
Use video and audio at least 20 seconds long. Wan may trim a few frames; short inputs are not extended.

The source orientation and aspect ratio determine the output dimensions:
- **720p mode:** longest side 1280 px (16:9 → 1280×720; 9:16 → 720×1280; square → 1280×1280).
- **480p mode:** longest side 832 px (16:9 → 832×464; 9:16 → 464×832; square → 832×832).

Both dimensions are rounded to multiples of 16. This may crop a few edge pixels.
The exact output size is shown in step 7. Changing video or resolution resets face selection.
Square frames contain more pixels than widescreen frames at the same longest side and may use more memory.

The v5.4 window-alignment fix and sampler settings are retained: window 121, motion 9, drop 8.
A user confirmed a successful 10-second v5.4 test; 20-second generation and these new shapes still need GPU validation.

Face selection now includes tracks detected in any frame. Track IDs are preserved when other faces disappear.
For frames without the selected track, the mask is empty instead of falling back to another face or all faces.
Run Detect faces again in this new runtime and choose the number shown in its preview.
Tracking can still lose a person or assign a new track after an interruption; inspect several frames before generation.

Progress bars cover Detect faces, Preview selected face and Generate video, including recovery.
Overall progress is an estimate based on completed stages; it is not a time estimate.
Frame/step counts drive the current-stage bar. Loading and individual sampling steps can pause the bar.
100% is displayed only after the server reports successful completion.
Audio settings: **audio_scale = 1.2**, **audio_cfg_scale = 1.2**.


In [ ]:
#@title 1. Check GPU and storage
from pathlib import Path
import os, sys, subprocess, shutil, json, hashlib, time
ROOT = Path('/content/pick-face')
ROOT.mkdir(parents=True, exist_ok=True)
PORT = 8188
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], check=True)
mem = next(line for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:'))
print('RAM:', round(int(mem.split()[1]) / 1024**2, 1), 'GiB')
print('Free disk:', round(shutil.disk_usage(ROOT).free / 1e9, 1), 'GB')
print('Models will be downloaded directly. Ready for setup.')


In [ ]:
#@title 2. Download the source archive from GitHub
import io, tarfile, urllib.request

SOURCE_URL = 'https://github.com/nikitalisin/pick-face-colab/releases/download/sources-v1/pick_face_sources_public_v1.tar.gz'
EXPECTED_ARCHIVE_SHA256 = 'feaac41ef22b9e5db71e825979a9fc6035cb380a154005cd1f4670cd44e4f403'
payload = ROOT / 'payload'

def safe_extract(archive, destination):
    destination = destination.resolve()
    for member in archive.getmembers():
        target = (destination / member.name).resolve()
        if not target.is_relative_to(destination) or not (member.isfile() or member.isdir()):
            raise RuntimeError('Invalid archive path or file type: ' + member.name)
    archive.extractall(destination)

if not (payload / '.ready').exists():
    print('Downloading source files from GitHub...', flush=True)
    request = urllib.request.Request(SOURCE_URL, headers={'User-Agent': 'PickFace-Colab'})
    with urllib.request.urlopen(request, timeout=120) as response:
        raw = response.read()
    if hashlib.sha256(raw).hexdigest() != EXPECTED_ARCHIVE_SHA256:
        raise RuntimeError('Source archive checksum mismatch. No files were extracted.')
    with tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz') as archive:
        safe_extract(archive, payload)
    expected = json.loads((payload / 'source-checksums.json').read_text())
    for name, digest in expected.items():
        if hashlib.sha256((payload / name).read_bytes()).hexdigest() != digest:
            raise RuntimeError('Source file checksum mismatch: ' + name)
    (payload / '.ready').write_text(EXPECTED_ARCHIVE_SHA256)
elif (payload / '.ready').read_text() != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError('This folder contains a different snapshot. Use a fresh Colab runtime.')
print('Source files are ready and verified. No manual archive upload is needed.')


In [ ]:
#@title 3. Prepare the bundled dependency
import base64
_COMFY_ENV_WHEEL = (
    '...'
    'BQYAAAAAJgAmAHsLAADmNQEAAAA='
)
_wheel_bytes = base64.b64decode(_COMFY_ENV_WHEEL, validate=True)
assert hashlib.sha256(_wheel_bytes).hexdigest() == '9aef9c572739e2bd3a46cafbd80b4fa1cebfd2f1eb8c3177ffffbd6b37c6d346', 'Bundled comfy-env checksum mismatch'
COMFY_ENV_WHEEL = ROOT / 'comfy_env-0.2.10-py3-none-any.whl'
COMFY_ENV_WHEEL.write_bytes(_wheel_bytes)
del _COMFY_ENV_WHEEL, _wheel_bytes
print('Bundled comfy-env 0.2.10 is ready and verified.')


In [ ]:
#@title 4. Install the runtime and ComfyUI
import queue, threading, signal
INSTALL_LOG = ROOT / 'install-v2.log'

def run_visible(args, title, env=None):
    print('\n' + title, flush=True)
    messages = queue.Queue()
    started = time.monotonic()
    proc = subprocess.Popen([str(x) for x in args], env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, start_new_session=True)
    def read_output():
        try:
            for line in proc.stdout:
                messages.put(line)
        finally:
            messages.put(None)
    threading.Thread(target=read_output, daemon=True).start()
    try:
        with INSTALL_LOG.open('a') as log:
            log.write('\n' + title + '\n')
            while True:
                try:
                    line = messages.get(timeout=20)
                except queue.Empty:
                    print(f'Running: {title} ({int(time.monotonic()-started)} seconds)', flush=True)
                    continue
                if line is None:
                    break
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
        status = proc.wait()
    except BaseException:
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()
        raise
    if status:
        raise RuntimeError(f'{title}: exit code {status}. See the error above. Log: {INSTALL_LOG}')
    print('Done:', title, flush=True)

RUN_ENV = os.environ.copy()
RUN_ENV.update(USE_COMFY_ENV='0', PYTHONUNBUFFERED='1',
               PIP_DISABLE_PIP_VERSION_CHECK='1', PIP_PROGRESS_BAR='off',
               PIP_DEFAULT_TIMEOUT='120', PIP_RETRIES='5', DEBIAN_FRONTEND='noninteractive')
run_visible([sys.executable, '-m', 'pip', 'install', 'uv==0.10.3'], '1/5: uv', RUN_ENV)
run_visible(['apt-get', 'update'], '2/5: system package index', RUN_ENV)
run_visible(['apt-get', 'install', '-y', 'git', 'ffmpeg', 'libgl1', 'libglib2.0-0'],
            '3/5: system packages', RUN_ENV)
uv = [sys.executable, '-m', 'uv']
PYTHON = ROOT / 'venv/bin/python'
if not PYTHON.exists():
    run_visible(uv + ['python', 'install', '3.10.19'], '4/5: Python 3.10.19', RUN_ENV)
    run_visible(uv + ['venv', '--seed', '--python', '3.10.19', str(ROOT / 'venv')],
                '4/5: isolated environment', RUN_ENV)
RUN_ENV['PATH'] = str(PYTHON.parent) + os.pathsep + RUN_ENV.get('PATH', '')
HELPER = ROOT / 'payload/support/runtime-v2.py'
HELPER.write_text('"""Reproduce the saved ComfyUI installation without touching an existing runtime."""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef run(args, **kwargs):\n    print(\'>>>\', \' \'.join(str(x) for x in args), flush=True)\n    subprocess.run([str(x) for x in args], check=True, **kwargs)\n\n\ndef checkout(url, commit, destination):\n    if (destination / \'.git\').exists():\n        result = subprocess.run([\'git\', \'-C\', str(destination), \'rev-parse\', \'HEAD\'],\n                                text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n        if result.returncode == 0:\n            if result.stdout.strip() != commit:\n                raise RuntimeError(f\'Different version at {destination}. Refusing to overwrite it.\')\n            return\n        origin = subprocess.check_output([\'git\', \'-C\', str(destination), \'remote\', \'get-url\', \'origin\'], text=True).strip()\n        if origin != url:\n            raise RuntimeError(f\'Different repository origin: {destination}\')\n    else:\n        if destination.exists() and any(destination.iterdir()):\n            raise RuntimeError(f\'Nonempty folder without git: {destination}\')\n        destination.mkdir(parents=True, exist_ok=True)\n        run([\'git\', \'init\', \'-q\', destination])\n        run([\'git\', \'-C\', destination, \'remote\', \'add\', \'origin\', url])\n    run([\'git\', \'-C\', destination, \'fetch\', \'--depth\', \'1\', \'origin\', commit])\n    run([\'git\', \'-C\', destination, \'checkout\', \'--detach\', \'FETCH_HEAD\'])\n\n\ndef install(root):\n    payload = root / \'payload\'\n    core = root / \'ComfyUI\'\n    manifest = json.loads((payload / \'manifest.json\').read_text())\n    checkout(\'https://github.com/Comfy-Org/ComfyUI.git\', manifest[\'comfyui_commit\'], core)\n    for name in [\'ComfyUI-KJNodes\', \'ComfyUI-MelBandRoFormer\', \'ComfyUI-VideoHelperSuite\', \'ComfyUI-WanVideoWrapper\']:\n        repo = manifest[\'repositories\'][name]\n        checkout(repo[\'remote\'], repo.get(\'upstream_base\') or repo[\'head\'], core / \'custom_nodes\' / name)\n\n    marker = core / \'.pick-face-snapshot\'\n    if not marker.exists():\n        for name in [\'custom_nodes\', \'input\']:\n            shutil.copytree(payload / name, core / name, dirs_exist_ok=True)\n        workflows = core / \'user\' / \'default\' / \'workflows\'\n        workflows.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(payload / \'workflows/WF_pick_face_original.json\', workflows / \'WF_pick_face_original.json\')\n        marker.write_text(manifest[\'snapshot\'])\n    # The original sources are never overwritten on subsequent runs.\n    pip = [sys.executable, \'-m\', \'pip\']\n    run(pip + [\'install\', \'torch==2.10.0\', \'torchvision==0.25.0\', \'torchaudio==2.10.0\',\n               \'--index-url\', \'https://download.pytorch.org/whl/cu128\'])\n    run(pip + [\'install\', \'--no-deps\', \'-r\', str(payload / \'requirements-runtime.lock.txt\')])\n    wheels = json.loads((payload / \'support/cuda-wheels-v2.json\').read_text())\n    for name, wheel in wheels.items():\n        print(\'CUDA extension:\', name, wheel[\'version\'], flush=True)\n        run(pip + [\'install\', \'--no-deps\', wheel[\'url\']])\n    run(pip + [\'check\'])\n    import torch\n    print(\'Python:\', sys.version.split()[0], \'Torch:\', torch.__version__, \'CUDA:\', torch.version.cuda)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA is unavailable. Select a Colab GPU runtime and repeat setup.\')\n    print(\'GPU:\', torch.cuda.get_device_name(0))\n    # Verify the installed accelerator on the actual Colab GPU before loading 49 GB of weights.\n    from sageattention import sageattn\n    q = torch.randn(1, 4, 64, 64, device=\'cuda\', dtype=torch.float16)\n    out = sageattn(q, q, q)\n    torch.cuda.synchronize()\n    if not torch.isfinite(out).all():\n        raise RuntimeError(\'SageAttention returned nonfinite values.\')\n    print(\'SageAttention check passed. Original workflow saved.\', flush=True)\n\n\ndef download_one(item, destination, token):\n    import requests\n    from tqdm import tqdm\n    if destination.exists():\n        if destination.stat().st_size == item[\'bytes\'] and sha256(destination) == item[\'sha256\']:\n            print(\'Already verified:\', item[\'path\'], flush=True)\n            return\n        raise RuntimeError(f\'Unexpected file at {destination}. Rename it before downloading again.\')\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_name(destination.name + \'.part\')\n    for attempt in range(5):\n        offset = partial.stat().st_size if partial.exists() else 0\n        if offset == item[\'bytes\']:\n            break\n        if offset > item[\'bytes\']:\n            raise RuntimeError(f\'Invalid partial file size: {partial}\')\n        headers = {\'Accept-Encoding\': \'identity\'}\n        if token:\n            headers[\'Authorization\'] = \'Bearer \' + token\n        if offset:\n            headers[\'Range\'] = f\'bytes={offset}-\'\n        try:\n            # Requests strips Authorization when following a redirect to another host.\n            with requests.get(item[\'url\'], headers=headers, stream=True, timeout=(30, 180)) as response:\n                response.raise_for_status()\n                if response.status_code == 206:\n                    content_range = response.headers.get(\'Content-Range\', \'\')\n                    if not content_range.startswith(f\'bytes {offset}-\'):\n                        raise RuntimeError(\'Server returned an invalid download range.\')\n                else:\n                    offset = 0\n                with partial.open(\'ab\' if offset else \'wb\') as f, tqdm(\n                    total=item[\'bytes\'], initial=offset, unit=\'B\', unit_scale=True,\n                    desc=destination.name,\n                ) as progress:\n                    for chunk in response.iter_content(8 * 1024 * 1024):\n                        if chunk:\n                            f.write(chunk)\n                            progress.update(len(chunk))\n            break\n        except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError):\n            if attempt == 4:\n                raise\n            time.sleep(2 ** attempt)\n    if partial.stat().st_size != item[\'bytes\'] or sha256(partial) != item[\'sha256\']:\n        raise RuntimeError(f\'SHA-256 verification failed: {partial}. File will not be used.\')\n    partial.replace(destination)\n\n\ndef download(root, infinite_source, infinite_path):\n    entries = json.loads((root / \'payload/support/models.json\').read_text())\n    target = root / \'ComfyUI/models\'\n    token = os.environ.get(\'HF_TOKEN\', \'\')\n    selected = [dict(x) for x in entries]\n    infinite = next(x for x in selected if x[\'path\'].endswith(\'/infinitetalk_single.safetensors\'))\n    if infinite_source == \'drive\':\n        infinite[\'sha256\'] = infinite[\'local_sha256\']\n        infinite[\'matches_local\'] = True\n        destination = target / infinite[\'path\']\n        if not (destination.exists() and sha256(destination) == infinite[\'sha256\']):\n            source = Path(infinite_path)\n            if not source.is_file():\n                raise FileNotFoundError(f\'Upload your infinitetalk_single.safetensors to Drive: {source}\')\n            if source.stat().st_size != infinite[\'bytes\'] or sha256(source) != infinite[\'sha256\']:\n                raise RuntimeError(\'InfiniteTalk on Drive does not match the original SHA-256.\')\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            if destination.exists():\n                raise RuntimeError(f\'A different version exists at {destination}. Rename it before replacing.\')\n            partial = destination.with_name(destination.name + \'.part\')\n            shutil.copyfile(source, partial)\n            if sha256(partial) != infinite[\'sha256\']:\n                raise RuntimeError(\'InfiniteTalk copy verification failed.\')\n            partial.replace(destination)\n        print(\'InfiniteTalk: original version from Drive.\', flush=True)\n    else:\n        print(\'InfiniteTalk: official MeiGen-AI download; weights differ from the original local copy.\', flush=True)\n    needed = sum(x[\'bytes\'] for x in selected if not (target / x[\'path\']).exists())\n    if shutil.disk_usage(root).free < needed + 5 * 1024 ** 3:\n        raise RuntimeError(f\'Insufficient disk space; additional space required: {needed / 1e9 + 5:.1f} GB.\')\n    for item in selected:\n        download_one(item, target / item[\'path\'], token)\n    (root / \'selected-models.json\').write_text(json.dumps(selected, ensure_ascii=False, indent=2))\n    print(\'All 13 files verified; 9 model components.\', flush=True)\n\n\ndef check(root, port):\n    import urllib.request\n    with urllib.request.urlopen(f\'http://127.0.0.1:{port}/object_info\', timeout=90) as response:\n        info = json.load(response)\n    workflow = json.loads((root / \'payload/workflows/WF_pick_face_original.json\').read_text())\n    required = {n[\'type\'] for n in workflow[\'nodes\']}\n    missing = sorted(required - set(info))\n    report = {\'missing_nodes\': missing, \'required_classes\': len(required), \'registered_classes\': len(info)}\n    (root / \'node-check.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2))\n    if missing:\n        raise RuntimeError(\'Required nodes failed to load: \' + \', \'.join(missing) + \'. See comfyui.log.\')\n    # Model values must also be present in the loaders\' dropdowns.\n    checks = [(\'WanVideoModelLoader\', \'model\', \'wan2.1_i2v_480p_14B_fp16.safetensors\'),\n              (\'MultiTalkModelLoader\', \'model\', \'infinitetalk_single.safetensors\'),\n              (\'MelBandRoFormerModelLoader\', \'model_name\', \'MelBandRoformer_fp16.safetensors\')]\n    for cls, key, filename in checks:\n        inputs = info[cls][\'input\']\n        field = inputs.get(\'required\', {}).get(key, inputs.get(\'optional\', {}).get(key))\n        if field and isinstance(field[0], list) and filename not in field[0]:\n            raise RuntimeError(f\'{cls}: model {filename} was not found in the list.\')\n    print(f\'All {len(required)} original workflow node types are available.\', flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'action\', choices=[\'install\', \'download\', \'check\'])\n    parser.add_argument(\'--root\', type=Path, required=True)\n    parser.add_argument(\'--infinite-source\', choices=[\'drive\', \'url\'], default=\'drive\')\n    parser.add_argument(\'--infinite-path\', default=\'\')\n    parser.add_argument(\'--port\', type=int, default=8188)\n    args = parser.parse_args()\n    if args.action == \'install\':\n        install(args.root)\n    elif args.action == \'download\':\n        download(args.root, args.infinite_source, args.infinite_path)\n    else:\n        check(args.root, args.port)\n', encoding='utf-8')
(ROOT / 'payload/support/cuda-wheels-v2.json').write_text('{"sageattention": {"version": "2.2.0+cu128torch2.10", "url": "https://github.com/PozzettiAndrea/cuda-wheels/releases/download/sageattention-latest/sageattention-2.2.0%2Bcu128torch2.10-cp310-cp310-manylinux_2_34_x86_64.manylinux_2_35_x86_64.whl#sha256=b9d0fa5ab91590a181294470217d352c650ad0ec07b954320e8eb9ba6c7c4b25"}, "flash-attn": {"version": "2.8.3+cu128torch2.10", "url": "https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.7.12/flash_attn-2.8.3%2Bcu128torch2.10-cp310-cp310-linux_x86_64.whl"}, "cc-torch": {"version": "0.2+cu128torch2.10", "url": "https://github.com/PozzettiAndrea/cuda-wheels/releases/download/cc_torch-latest/cc_torch-0.2%2Bcu128torch2.10-cp310-cp310-linux_x86_64.whl#sha256=a9d74e8f6ba981a8dfd5f2a600018be6bd836e97a94e1748db1b2aba72f0903e"}, "torch-generic-nms": {"version": "0.1+cu128torch2.10", "url": "https://github.com/PozzettiAndrea/cuda-wheels/releases/download/torch_generic_nms-latest/torch_generic_nms-0.1%2Bcu128torch2.10-cp310-cp310-manylinux_2_34_x86_64.manylinux_2_35_x86_64.whl#sha256=a76b04931ff5e58cb9f30b858587c2a8146c1eb93c91da3caf9d4714020d8ceb"}}', encoding='utf-8')
run_visible([str(PYTHON), '-m', 'pip', 'install', '--no-deps', str(COMFY_ENV_WHEEL)],
            'Bundled comfy-env 0.2.10', RUN_ENV)
run_visible([str(PYTHON), '-u', str(HELPER), 'install', '--root', str(ROOT)],
            '5/5: ComfyUI, dependencies, and GPU checks', RUN_ENV)


In [ ]:
#@title 5. Download and verify models
run_visible([str(PYTHON), '-u', str(HELPER), 'download', '--root', str(ROOT),
             '--infinite-source', 'url'], 'Downloading and verifying models', RUN_ENV)


In [ ]:
#@title 6. Start the generation server
import urllib.request, urllib.error, socket
(ROOT / 'prepare-v4.py').write_text('"""Prepare only the Colab copy; keep the original graph and source backups."""\nimport ast\nimport hashlib\nimport json\nfrom pathlib import Path\n\nCHECK = \'\'\'\n# pick-face-v4 finite checks\n def_placeholder\n\'\'\'.replace(\' def_placeholder\', \'\'\'def _pick_face_finite(value, stage):\n    if isinstance(value, torch.Tensor):\n        flat = value.detach().reshape(-1)\n        for part in flat.split(1048576):\n            if not torch.isfinite(part).all().item():\n                raise RuntimeError(f"[PickFace] NaN/Inf at {stage}; dtype={value.dtype}, shape={tuple(value.shape)}. Generation stopped before saving invalid video.")\n    elif isinstance(value, (list, tuple)):\n        for item in value:\n            _pick_face_finite(item, stage)\n    return value\n\'\'\')\n\n\ndef instrument_sampler(source):\n    tree = ast.parse(source)\n    lines = source.splitlines(keepends=True)\n    inserts = []\n    for n in ast.walk(tree):\n        if isinstance(n, ast.Assign) and isinstance(n.value, ast.Call) and isinstance(n.value.func, ast.Name) and n.value.func.id in (\'predict_with_cfg\', \'predict_func\'):\n            target = n.targets[0]\n            if isinstance(target, ast.Tuple) and isinstance(target.elts[0], ast.Name):\n                inserts.append((n.end_lineno, \' \' * n.col_offset + f\'_pick_face_finite({target.elts[0].id}, "diffusion prediction")\\n\'))\n    if not inserts:\n        raise RuntimeError(\'Sampler source changed: no predictions found\')\n    for index, code in sorted(inserts, reverse=True):\n        lines.insert(index, code)\n    result = \'\'.join(lines) + CHECK\n    ast.parse(result)\n    return result\n\n\ndef instrument_vae(source):\n    a = \'        hidden_states = [hidden_state.to("cpu") for hidden_state in hidden_states]\'\n    b = \'            videos.append(video)\'\n    c = \'        return hidden_states\'\n    assert source.count(a) == source.count(b) == source.count(c) == 1\n    source = source.replace(a, \'        _pick_face_finite(hidden_states, "VAE decode input")\\n\' + a)\n    source = source.replace(b, \'            _pick_face_finite(video, "VAE decode output")\\n\' + b)\n    source = source.replace(c, \'        _pick_face_finite(hidden_states, "VAE encode output")\\n\' + c)\n    result = source + CHECK\n    ast.parse(result)\n    return result\n\n\ndef prepare(root):\n    root = Path(root)\n    core = root / \'ComfyUI\'\n    wrapper = core / \'custom_nodes/ComfyUI-WanVideoWrapper\'\n    report = {}\n    for relative, transform in [(\'nodes_sampler.py\', instrument_sampler), (\'multitalk/multitalk_loop.py\', instrument_sampler), (\'wanvideo/wan_video_vae.py\', instrument_vae)]:\n        p = wrapper / relative\n        original = root / \'payload/custom_nodes/ComfyUI-WanVideoWrapper\' / relative\n        if original.exists():\n            old = original.read_text()\n        else:\n            import subprocess\n            manifest = json.loads((root / \'payload/manifest.json\').read_text())\n            repo = manifest[\'repositories\'][\'ComfyUI-WanVideoWrapper\']\n            commit = repo.get(\'upstream_base\') or repo[\'head\']\n            old = subprocess.check_output([\'git\', \'-C\', str(wrapper), \'show\', f\'{commit}:{relative}\'], text=True)\n        new = transform(old)\n        current = p.read_text()\n        if current not in (old, new):\n            raise RuntimeError(f\'Unexpected local changes in {p}; not overwriting\')\n        backup = p.with_suffix(p.suffix + \'.pre-v4\')\n        if not backup.exists():\n            backup.write_text(old)\n        p.write_text(new)\n        report[relative] = hashlib.sha256(new.encode()).hexdigest()\n    (root / \'v4-diagnostics.json\').write_text(json.dumps(report, indent=2))\n    print(\'NaN/Inf checks installed: VAE encoding, diffusion prediction, VAE decoding.\')\n\n\ndef workflow(root, video=None, audio=None, profile=\'bf16_sdpa\'):\n    root = Path(root)\n    w = json.loads((root / \'payload/workflows/WF_pick_face_original.json\').read_text())\n    for n in w[\'nodes\']:\n        if n[\'id\'] == 374 and profile == \'bf16_sdpa\':\n            n[\'widgets_values\'][1] = \'bf16\'\n            n[\'widgets_values\'][4] = \'sdpa\'\n        if n[\'id\'] == 373 and video:\n            n[\'widgets_values\'][\'video\'] = video\n            n[\'widgets_values\'].pop(\'videopreview\', None)\n        if n[\'id\'] == 377 and audio:\n            n[\'widgets_values\'][\'audio\'] = audio\n            n[\'widgets_values\'].pop(\'audiopreview\', None)\n    destination = root / \'ComfyUI/user/default/workflows/WF_pick_face_Colab_v4.json\'\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    destination.write_text(json.dumps(w, ensure_ascii=False, indent=2))\n    return destination\n\nif __name__ == \'__main__\':\n    import sys\n    prepare(sys.argv[1])\n')
(ROOT / 'controller-v5.py').write_text('"""Colab-side API client. No browser tab or public tunnel is required."""\nimport copy\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\nfrom urllib.request import Request, urlopen\nfrom urllib.error import HTTPError\nfrom urllib.parse import urlencode\n\n\ndef ancestors(graph, targets):\n    keep = set()\n    def visit(node):\n        if node in keep:\n            return\n        keep.add(node)\n        for value in graph[node][\'inputs\'].values():\n            if isinstance(value, list) and len(value) == 2 and str(value[0]) in graph:\n                visit(str(value[0]))\n    for node in targets:\n        visit(node)\n    return {key: value for key, value in graph.items() if key in keep}\n\n\ndef make_prompt(template, video, audio, face_id, mode):\n    graph = copy.deepcopy(template)\n    graph[\'373\'][\'inputs\'][\'video\'] = video\n    graph[\'377\'][\'inputs\'][\'audio\'] = audio\n    graph[\'374\'][\'inputs\'].update(base_precision=\'bf16\', attention_mode=\'sdpa\')\n    graph[\'600\'] = {\'class_type\': \'PickFaceSelectionInfo\', \'inputs\': {\n        \'masks\': [\'502\', 0], \'selected_id\': -1 if mode == \'detect\' else int(face_id)}}\n    graph[\'503\'][\'inputs\'].update(masks=[\'600\', 0],\n        obj_id=-1 if mode == \'detect\' else int(face_id), plot_all_masks=mode == \'detect\')\n    if mode not in (\'detect\', \'preview\', \'generate\'):\n        raise ValueError(\'Unknown operation\')\n    return ancestors(graph, [\'131\'] if mode == \'generate\' else [\'510\'])\n\n\nclass Client:\n    def __init__(self, root, port=8188):\n        self.root = Path(root)\n        self.base = f\'http://127.0.0.1:{port}\'\n        self.state_path = self.root / \'colab-v5-state.json\'\n        self.state = json.loads(self.state_path.read_text()) if self.state_path.exists() else {}\n\n    def api(self, path, data=None):\n        body = None if data is None else json.dumps(data).encode()\n        req = Request(self.base + path, data=body, headers={\'Content-Type\': \'application/json\'})\n        try:\n            with urlopen(req, timeout=30) as response:\n                return json.load(response)\n        except HTTPError as exc:\n            raise RuntimeError(f\'ComfyUI HTTP {exc.code}: {exc.read().decode()[:5000]}\') from exc\n\n    def save(self):\n        self.state_path.write_text(json.dumps(self.state, indent=2))\n\n    def submit(self, graph, mode, context):\n        q = self.api(\'/queue\')\n        if q.get(\'queue_running\') or q.get(\'queue_pending\'):\n            raise RuntimeError(\'ComfyUI is busy. Wait for the current job, then use Recover last job.\')\n        # Persist the exact submitted graph for diagnostics; no UI workflow conversion.\n        (self.root / \'last-prompt-v5.json\').write_text(json.dumps(graph, indent=2))\n        result = self.api(\'/prompt\', {\'prompt\': graph, \'client_id\': str(uuid.uuid4())})\n        if result.get(\'node_errors\') or \'prompt_id\' not in result:\n            raise RuntimeError(json.dumps(result))\n        self.state = {\'prompt_id\': result[\'prompt_id\'], \'mode\': mode, \'context\': context}\n        self.save()\n        return self.state[\'prompt_id\']\n\n    def wait(self, report=print):\n        prompt_id = self.state.get(\'prompt_id\')\n        if not prompt_id:\n            raise RuntimeError(\'No submitted job to recover.\')\n        started = time.monotonic()\n        while True:\n            entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n            if entry:\n                status = entry.get(\'status\', {})\n                if status.get(\'status_str\') == \'error\':\n                    details = [v.get(\'exception_message\', kind) for kind, v in status.get(\'messages\', [])\n                               if kind in (\'execution_error\', \'execution_interrupted\')]\n                    raise RuntimeError(\'\\n\'.join(details) or \'Job failed. Download diagnostics in step 9.\')\n                if status.get(\'completed\'):\n                    return entry\n            q = self.api(\'/queue\')\n            jobs = q.get(\'queue_running\', []) + q.get(\'queue_pending\', [])\n            if not entry and not any(item[1] == prompt_id for item in jobs):\n                # History can become available between the two requests.\n                entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n                if not entry:\n                    raise RuntimeError(\'Job is no longer in the queue or history. Detect faces again.\')\n            report(f\'{self.state["mode"].capitalize()} in progress — {int(time.monotonic()-started)} seconds\')\n            time.sleep(3)\n\n    def image(self, item):\n        query = urlencode({k: item[k] for k in (\'filename\', \'subfolder\', \'type\')})\n        with urlopen(self.base + \'/view?\' + query, timeout=30) as response:\n            return response.read()\n')
(ROOT / 'prompt-v5.json').write_text('{\n  "120": {\n    "inputs": {\n      "model": "infinitetalk_single.safetensors"\n    },\n    "class_type": "MultiTalkModelLoader"\n  },\n  "128": {\n    "inputs": {\n      "steps": 4,\n      "cfg": 1.0,\n      "shift": 11.0,\n      "seed": 465484085186226,\n      "force_offload": true,\n      "scheduler": "dpm++_sde",\n      "riflex_freq_index": 0,\n      "denoise_strength": 1.0,\n      "batched_cfg": false,\n      "rope_function": "comfy",\n      "start_step": 3,\n      "end_step": -1,\n      "add_noise_to_samples": true,\n      "model": [\n        "508",\n        0\n      ],\n      "image_embeds": [\n        "192",\n        0\n      ],\n      "text_embeds": [\n        "241",\n        0\n      ],\n      "samples": [\n        "229",\n        0\n      ],\n      "multitalk_embeds": [\n        "194",\n        0\n      ]\n    },\n    "class_type": "WanVideoSampler"\n  },\n  "129": {\n    "inputs": {\n      "model_name": "Wan2_1_VAE_bf16.safetensors",\n      "precision": "bf16",\n      "use_cpu_cache": false,\n      "verbose": false\n    },\n    "class_type": "WanVideoVAELoader"\n  },\n  "130": {\n    "inputs": {\n      "enable_vae_tiling": false,\n      "tile_x": 272,\n      "tile_y": 272,\n      "tile_stride_x": 144,\n      "tile_stride_y": 128,\n      "normalization": "default",\n      "vae": [\n        "129",\n        0\n      ],\n      "samples": [\n        "128",\n        0\n      ]\n    },\n    "class_type": "WanVideoDecode"\n  },\n  "131": {\n    "inputs": {\n      "frame_rate": [\n        "307",\n        0\n      ],\n      "loop_count": 0,\n      "filename_prefix": "pick_face",\n      "format": "video/h264-mp4",\n      "pix_fmt": "yuv420p",\n      "crf": 19,\n      "save_metadata": true,\n      "trim_to_audio": true,\n      "pingpong": false,\n      "save_output": true,\n      "images": [\n        "301",\n        0\n      ],\n      "audio": [\n        "194",\n        1\n      ]\n    },\n    "class_type": "VHS_VideoCombine"\n  },\n  "134": {\n    "inputs": {\n      "blocks_to_swap": 40,\n      "offload_img_emb": true,\n      "offload_txt_emb": true,\n      "use_non_blocking": true,\n      "vace_blocks_to_swap": 0,\n      "prefetch_blocks": 1,\n      "block_swap_debug": false\n    },\n    "class_type": "WanVideoBlockSwap"\n  },\n  "137": {\n    "inputs": {\n      "model": "TencentGameMate/chinese-wav2vec2-base",\n      "base_precision": "fp16",\n      "load_device": "main_device"\n    },\n    "class_type": "DownloadAndLoadWav2VecModel"\n  },\n  "192": {\n    "inputs": {\n      "width": [\n        "291",\n        1\n      ],\n      "height": [\n        "291",\n        2\n      ],\n      "frame_window_size": 121,\n      "motion_frame": 9,\n      "force_offload": true,\n      "colormatch": "disabled",\n      "tiled_vae": false,\n      "mode": "infinitetalk",\n      "output_path": "",\n      "vae": [\n        "129",\n        0\n      ],\n      "start_image": [\n        "291",\n        0\n      ],\n      "clip_embeds": [\n        "237",\n        0\n      ]\n    },\n    "class_type": "WanVideoImageToVideoMultiTalk"\n  },\n  "194": {\n    "inputs": {\n      "normalize_loudness": true,\n      "num_frames": [\n        "373",\n        1\n      ],\n      "fps": [\n        "307",\n        0\n      ],\n      "audio_scale": 1.5,\n      "audio_cfg_scale": 1.5,\n      "multi_audio_type": "para",\n      "audio_token_gate": true,\n      "add_noise_floor": false,\n      "smooth_transients": false,\n      "wav2vec_model": [\n        "137",\n        0\n      ],\n      "audio_1": [\n        "369",\n        0\n      ]\n    },\n    "class_type": "MultiTalkWav2VecEmbeds"\n  },\n  "229": {\n    "inputs": {\n      "enable_vae_tiling": false,\n      "tile_x": 272,\n      "tile_y": 272,\n      "tile_stride_x": 160,\n      "tile_stride_y": 128,\n      "noise_aug_strength": 0.0,\n      "latent_strength": 1.0,\n      "vae": [\n        "129",\n        0\n      ],\n      "image": [\n        "230",\n        0\n      ]\n    },\n    "class_type": "WanVideoEncode"\n  },\n  "230": {\n    "inputs": {\n      "width": [\n        "245",\n        0\n      ],\n      "height": [\n        "246",\n        0\n      ],\n      "upscale_method": "lanczos",\n      "keep_proportion": "crop",\n      "pad_color": "0, 0, 0",\n      "crop_position": "center",\n      "divisible_by": 16,\n      "device": "cpu",\n      "image": [\n        "373",\n        0\n      ]\n    },\n    "class_type": "ImageResizeKJv2"\n  },\n  "231": {\n    "inputs": {\n      "start_index": 0,\n      "num_frames": 1,\n      "images": [\n        "230",\n        0\n      ]\n    },\n    "class_type": "GetImageRangeFromBatch"\n  },\n  "237": {\n    "inputs": {\n      "strength_1": 1.0,\n      "strength_2": 1.0,\n      "crop": "center",\n      "combine_embeds": "average",\n      "force_offload": true,\n      "tiles": 0,\n      "ratio": 0.5,\n      "clip_vision": [\n        "238",\n        0\n      ],\n      "image_1": [\n        "291",\n        0\n      ]\n    },\n    "class_type": "WanVideoClipVisionEncode"\n  },\n  "238": {\n    "inputs": {\n      "clip_name": "clip_vision_h.safetensors"\n    },\n    "class_type": "CLIPVisionLoader"\n  },\n  "241": {\n    "inputs": {\n      "model_name": "umt5-xxl-enc-fp8_e4m3fn.safetensors",\n      "precision": "bf16",\n      "positive_prompt": "person talking",\n      "negative_prompt": "bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards",\n      "quantization": "fp8_e4m3fn",\n      "use_disk_cache": false,\n      "device": "gpu"\n    },\n    "class_type": "WanVideoTextEncodeCached"\n  },\n  "245": {\n    "inputs": {\n      "value": 832\n    },\n    "class_type": "INTConstant"\n  },\n  "246": {\n    "inputs": {\n      "value": 480\n    },\n    "class_type": "INTConstant"\n  },\n  "270": {\n    "inputs": {\n      "value": 120\n    },\n    "class_type": "INTConstant"\n  },\n  "291": {\n    "inputs": {\n      "image": [\n        "231",\n        0\n      ]\n    },\n    "class_type": "GetImageSizeAndCount"\n  },\n  "293": {\n    "inputs": {\n      "preview": "117",\n      "previewMode": null,\n      "source": [\n        "194",\n        2\n      ]\n    },\n    "class_type": "PreviewAny"\n  },\n  "299": {\n    "inputs": {\n      "inputcount": 2,\n      "direction": "down",\n      "match_image_size": false,\n      "Update inputs": null,\n      "image_1": [\n        "301",\n        0\n      ],\n      "image_2": [\n        "230",\n        0\n      ]\n    },\n    "class_type": "ImageConcatMulti"\n  },\n  "300": {\n    "inputs": {\n      "image": [\n        "130",\n        0\n      ]\n    },\n    "class_type": "GetImageSizeAndCount"\n  },\n  "301": {\n    "inputs": {\n      "start_index": 0,\n      "num_frames": [\n        "194",\n        2\n      ],\n      "images": [\n        "300",\n        0\n      ]\n    },\n    "class_type": "GetImageRangeFromBatch"\n  },\n  "307": {\n    "inputs": {\n      "video_info": [\n        "373",\n        3\n      ]\n    },\n    "class_type": "VHS_VideoInfo"\n  },\n  "315": {\n    "inputs": {\n      "frame_rate": 25.0,\n      "loop_count": 0,\n      "filename_prefix": "pick_face",\n      "format": "video/h264-mp4",\n      "pix_fmt": "yuv420p",\n      "crf": 19,\n      "save_metadata": true,\n      "trim_to_audio": true,\n      "pingpong": false,\n      "save_output": false,\n      "images": [\n        "299",\n        0\n      ],\n      "audio": [\n        "194",\n        1\n      ]\n    },\n    "class_type": "VHS_VideoCombine"\n  },\n  "318": {\n    "inputs": {\n      "images": [\n        "230",\n        0\n      ]\n    },\n    "class_type": "PreviewImage"\n  },\n  "319": {\n    "inputs": {\n      "images": [\n        "231",\n        0\n      ]\n    },\n    "class_type": "PreviewImage"\n  },\n  "348": {\n    "inputs": {\n      "images": [\n        "373",\n        0\n      ]\n    },\n    "class_type": "PreviewImage"\n  },\n  "369": {\n    "inputs": {\n      "model": [\n        "376",\n        0\n      ],\n      "audio": [\n        "377",\n        0\n      ]\n    },\n    "class_type": "MelBandRoFormerSampler"\n  },\n  "373": {\n    "inputs": {\n      "video": "UPLOAD_VIDEO.mp4",\n      "force_rate": 0.0,\n      "custom_width": [\n        "245",\n        0\n      ],\n      "custom_height": [\n        "246",\n        0\n      ],\n      "frame_load_cap": [\n        "270",\n        0\n      ],\n      "skip_first_frames": 0,\n      "select_every_nth": 1,\n      "format": "Wan"\n    },\n    "class_type": "VHS_LoadVideo"\n  },\n  "374": {\n    "inputs": {\n      "model": "wan2.1_i2v_480p_14B_fp16.safetensors",\n      "base_precision": "bf16",\n      "quantization": "fp8_e4m3fn",\n      "load_device": "offload_device",\n      "attention_mode": "sdpa",\n      "rms_norm_function": "pytorch",\n      "block_swap_args": [\n        "134",\n        0\n      ],\n      "lora": [\n        "375",\n        0\n      ],\n      "multitalk_model": [\n        "120",\n        0\n      ]\n    },\n    "class_type": "WanVideoModelLoader"\n  },\n  "375": {\n    "inputs": {\n      "lora": "Wan21_I2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors",\n      "strength": 1.0,\n      "low_mem_load": false,\n      "merge_loras": true\n    },\n    "class_type": "WanVideoLoraSelect"\n  },\n  "376": {\n    "inputs": {\n      "model_name": "MelBandRoformer_fp16.safetensors"\n    },\n    "class_type": "MelBandRoFormerModelLoader"\n  },\n  "377": {\n    "inputs": {\n      "audio": "UPLOAD_AUDIO.wav",\n      "start_time": 0.0,\n      "duration": 0.0\n    },\n    "class_type": "VHS_LoadAudioUpload"\n  },\n  "500": {\n    "inputs": {\n      "precision": "auto",\n      "attention": "auto",\n      "compile": false\n    },\n    "class_type": "LoadSAM3Model"\n  },\n  "501": {\n    "inputs": {\n      "prompt_mode": "text",\n      "text_prompt": "face",\n      "frame_idx": 0,\n      "score_threshold": 0.3,\n      "video_frames": [\n        "230",\n        0\n      ]\n    },\n    "class_type": "SAM3VideoSegmentation"\n  },\n  "502": {\n    "inputs": {\n      "start_frame": 1,\n      "end_frame": -1,\n      "direction": "forward",\n      "sam3_model": [\n        "500",\n        0\n      ],\n      "video_state": [\n        "501",\n        0\n      ]\n    },\n    "class_type": "SAM3Propagate"\n  },\n  "503": {\n    "inputs": {\n      "obj_id": 1,\n      "plot_all_masks": true,\n      "masks": [\n        "502",\n        0\n      ],\n      "video_state": [\n        "502",\n        2\n      ],\n      "scores": [\n        "502",\n        1\n      ]\n    },\n    "class_type": "SAM3VideoOutput"\n  },\n  "504": {\n    "inputs": {\n      "images": [\n        "503",\n        2\n      ]\n    },\n    "class_type": "PreviewImage"\n  },\n  "505": {\n    "inputs": {\n      "images": [\n        "503",\n        1\n      ]\n    },\n    "class_type": "PreviewImage"\n  },\n  "508": {\n    "inputs": {\n      "enabled": true,\n      "strength": 1.0,\n      "coverage_threshold": 0.5,\n      "background": "silence",\n      "motion_frame": 9,\n      "model": [\n        "374",\n        0\n      ],\n      "mask": [\n        "509",\n        0\n      ]\n    },\n    "class_type": "WanVideoAudioTokenGate"\n  },\n  "509": {\n    "inputs": {\n      "mouth_frac": 0.45,\n      "pad": 0,\n      "mask": [\n        "503",\n        0\n      ]\n    },\n    "class_type": "MaskToMouthBand"\n  },\n  "510": {\n    "inputs": {\n      "images": [\n        "503",\n        2\n      ]\n    },\n    "class_type": "PreviewImage"\n  },\n  "511": {\n    "inputs": {\n      "images": [\n        "503",\n        2\n      ]\n    },\n    "class_type": "PreviewImage"\n  }\n}')
(ROOT / 'ComfyUI/pickface_progress_server.py').write_text('"""Local read-only progress snapshot for Colab polling and reconnection."""\nfrom aiohttp import web\nfrom server import PromptServer\nfrom comfy_execution.progress import get_progress_state\n\n\n@PromptServer.instance.routes.get(\'/pickface/progress/{prompt_id}\')\nasync def progress_snapshot(request):\n    registry = get_progress_state()\n    if registry.prompt_id != request.match_info[\'prompt_id\']:\n        return web.json_response({\'prompt_id\': request.match_info[\'prompt_id\'], \'nodes\': {}})\n    states = {}\n    for key, entry in list(registry.nodes.items()):\n        entry = dict(entry)\n        states[str(key)] = {\'state\': entry[\'state\'].value,\n                            \'value\': entry[\'value\'], \'max\': entry[\'max\']}\n    return web.json_response({\'prompt_id\': registry.prompt_id, \'nodes\': states})\n')
FACE_NODE = ROOT / 'ComfyUI/custom_nodes/pick_face_colab_v5'
FACE_NODE.mkdir(parents=True, exist_ok=True)
(FACE_NODE / '__init__.py').write_text('"""Expose every detected track and prevent fallback to another face or union."""\nimport json\n\n\ndef normalized(frame):\n    shape = tuple(frame.shape)\n    if len(shape) == 4 and shape[0] == 1:\n        frame = frame[0]\n        shape = tuple(frame.shape)\n    if len(shape) != 3:\n        raise ValueError(\'SAM returned masks without an object axis. Detect faces again.\')\n    return frame\n\n\ndef inspect_masks(masks, selected_id=-1):\n    presence = {}\n    for frame in masks.values():\n        frame = normalized(frame)\n        for index in range(frame.shape[0]):\n            if bool(frame[index].any()):\n                presence[index] = presence.get(index, 0) + 1\n    ids = sorted(presence)\n    if selected_id >= 0 and selected_id not in ids:\n        raise ValueError(f\'Face ID {selected_id} was not detected. Valid IDs: {ids}\')\n    return {\'face_ids\': ids, \'frames\': len(masks),\n            \'visible_frames\': presence,\n            \'max_objects\': max(ids, default=-1) + 1}\n\n\nclass PickFaceSelectionInfo:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\'required\': {\'masks\': (\'SAM3_VIDEO_MASKS\',),\n                             \'selected_id\': (\'INT\', {\'default\': -1, \'min\': -1})}}\n\n    RETURN_TYPES = (\'SAM3_VIDEO_MASKS\',)\n    FUNCTION = \'inspect\'\n    OUTPUT_NODE = True\n    CATEGORY = \'PickFace\'\n\n    def inspect(self, masks, selected_id=-1):\n        info = inspect_masks(masks, selected_id)\n        result = masks\n        if selected_id >= 0:\n            # SAM3VideoOutput emits zero for absent frame keys. Never let it\n            # handle an out-of-range index (which selects a different/union mask).\n            result = {}\n            for key, frame in masks.items():\n                frame = normalized(frame)\n                if selected_id < frame.shape[0] and bool(frame[selected_id].any()):\n                    result[key] = frame\n        return {\'ui\': {\'text\': [json.dumps(info)]}, \'result\': (result,)}\n\n\nNODE_CLASS_MAPPINGS = {\'PickFaceSelectionInfo\': PickFaceSelectionInfo}\n\nimport pickface_progress_server\n')
import importlib.util
spec = importlib.util.spec_from_file_location('pick_face_v4', ROOT / 'prepare-v4.py')
pf4 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pf4)
(ROOT / 'window-patch-v5_4.py').write_text('"""Apply explicit mask-window synchronization only to the Colab copy."""\nimport ast\nfrom pathlib import Path\n\nSYNC = \'\'\'\ndef _pickface_sync_window(ctx, offset):\n    previous = ctx.get("explicit_start")\n    ctx["explicit_start"] = offset\n    if offset is not None and offset != previous:\n        ctx["window_idx"] = 0 if offset == 0 else ctx.get("window_idx", 0) + 1\n\'\'\'\n\n\ndef patch_gate(source):\n    old = \'        start = CTX["window_idx"] * max(frame_num - int(CTX["motion_frame"]), 1)\'\n    assert source.count(old) == 1\n    source = source.replace(old, \'        start = CTX.get("explicit_start")\\n\'\n        \'        if start is None:\\n    \' + old)\n    old = \'    def forward(self, *args, **kwargs):\\n        if CTX.get("gate") is not None:\'\n    assert source.count(old) == 1\n    source = source.replace(old, \'    def forward(self, *args, **kwargs):\\n\'\n        \'        _pickface_sync_window(CTX, getattr(self, "_pickface_source_start", None))\\n\'\n        \'        if CTX.get("gate") is not None:\')\n    source = source.replace(\'if step is not None and not bool(uncond):\',\n        \'if CTX.get("explicit_start") is None and step is not None and not bool(uncond):\')\n    source += SYNC\n    ast.parse(source)\n    return source\n\n\ndef patch_loop(source):\n    tree = ast.parse(source)\n    calls = [n for n in ast.walk(tree) if isinstance(n, ast.Assign)\n             and isinstance(n.value, ast.Call) and isinstance(n.value.func, ast.Name)\n             and n.value.func.id == \'predict_func\']\n    assert len(calls) == 1\n    n = calls[0]\n    lines = source.splitlines(keepends=True)\n    indent = \' \' * n.col_offset\n    replacement = [indent + \'transformer._pickface_source_start = int(audio_start_idx)\\n\',\n                   indent + \'try:\\n\']\n    replacement += [\'    \' + line for line in lines[n.lineno-1:n.end_lineno]]\n    replacement += [indent + \'finally:\\n\', indent + \'    del transformer._pickface_source_start\\n\']\n    lines[n.lineno-1:n.end_lineno] = replacement\n    result = \'\'.join(lines)\n    ast.parse(result)\n    return result\n\n\ndef restore_previous_patch(root):\n    base = Path(root) / \'ComfyUI/custom_nodes\'\n    for name, transform in [\n        (\'ComfyUI-WanAudioTokenGate/nodes.py\', patch_gate),\n        (\'ComfyUI-WanVideoWrapper/multitalk/multitalk_loop.py\', patch_loop)]:\n        p = base / name\n        marker = \'# pickface-v5.4-window-sync\\n\'\n        if marker not in p.read_text():\n            continue\n        old = p.with_suffix(\'.py.pre-v5_4\').read_text()\n        if p.read_text() != transform(old) + \'\\n\' + marker:\n            raise RuntimeError(\'Unexpected changes in \' + str(p))\n        p.write_text(old)\n\n\ndef prepare(root):\n    root = Path(root)\n    base = root / \'ComfyUI/custom_nodes\'\n    for name, transform in [\n        (\'ComfyUI-WanAudioTokenGate/nodes.py\', patch_gate),\n        (\'ComfyUI-WanVideoWrapper/multitalk/multitalk_loop.py\', patch_loop)]:\n        p = base / name\n        source = p.read_text()\n        marker = \'# pickface-v5.4-window-sync\\n\'\n        if marker in source:\n            continue\n        backup = p.with_suffix(\'.py.pre-v5_4\')\n        if not backup.exists():\n            backup.write_text(source)\n        p.write_text(transform(source) + \'\\n\' + marker)\n    print(\'Mask windows synchronized to the sampler source-frame offset.\')\n')
import importlib.util
window_spec = importlib.util.spec_from_file_location('pickface_window_patch', ROOT / 'window-patch-v5_4.py')
window_patch = importlib.util.module_from_spec(window_spec)
window_spec.loader.exec_module(window_patch)
(ROOT / 'sam-patch-v5_6.py').write_text('"""Preserve SAM track identity before exposing numbered faces."""\nimport ast\nfrom pathlib import Path\n\nHELPER = \'\'\'\n\ndef _pickface_stable_masks(mask, object_ids, slots):\n    if isinstance(mask, np.ndarray):\n        mask = torch.from_numpy(mask)\n    if mask.dim() == 4 and mask.shape[0] == 1:\n        mask = mask.squeeze(0)\n    if mask.dim() != 3:\n        raise ValueError("SAM masks must have an object axis")\n    if object_ids is None:\n        if mask.shape[0]:\n            raise ValueError("SAM did not return tracking IDs; refusing ambiguous face selection")\n        object_ids = []\n    ids = [int(value) for value in object_ids]\n    if len(ids) != mask.shape[0] or len(set(ids)) != len(ids):\n        raise ValueError("SAM tracking IDs do not match masks")\n    for track_id in ids:\n        if track_id not in slots:\n            slots[track_id] = len(slots)\n    stable = mask.new_zeros((len(slots), *mask.shape[1:]))\n    for row, track_id in enumerate(ids):\n        stable[slots[track_id]] = mask[row]\n    return stable\n\'\'\'\nMARKER = \'# pickface-v5.6-stable-sam-ids\'\n\n\ndef patch(source):\n    if MARKER in source:\n        return source\n    old = \'        masks_dict = {}\\n        scores_dict = {}\'\n    assert source.count(old) == 1\n    source = source.replace(old, old + \'\\n        pickface_slots = {}\')\n    old = \'                    masks_dict[frame_idx] = mask\'\n    assert source.count(old) == 1\n    source = source.replace(old, \'\'\'                    track_ids = outputs.get("out_obj_ids")\n                    if track_ids is None:\n                        track_ids = outputs.get("obj_ids")\n                    masks_dict[frame_idx] = _pickface_stable_masks(mask, track_ids, pickface_slots)\'\'\')\n    result = source + \'\\n\' + MARKER + \'\\n\' + HELPER\n    ast.parse(result)\n    return result\n\n\ndef prepare(root):\n    p = Path(root) / \'ComfyUI/custom_nodes/comfyui-sam3/nodes/sam3_video_nodes.py\'\n    source = p.read_text()\n    new = patch(source)\n    if new != source:\n        backup = p.with_suffix(\'.py.pre-v5_6\')\n        if not backup.exists():\n            backup.write_text(source)\n        p.write_text(new)\n    print(\'Stable SAM face IDs enabled; absent faces use empty masks.\')\n')
sam_spec = importlib.util.spec_from_file_location('pickface_sam_patch', ROOT / 'sam-patch-v5_6.py')
sam_patch = importlib.util.module_from_spec(sam_spec)
sam_spec.loader.exec_module(sam_patch)
(ROOT / 'progress-patch-v5_7.py').write_text('"""Add frame progress to the Colab SAM copy; no model/selection changes."""\nimport ast\nfrom pathlib import Path\nMARKER = \'# pickface-v5.7-frame-progress\'\n\ndef patch(source):\n    if MARKER in source:\n        return source\n    replacements = [\n        (\'        masks_dict = {}\', \'\'\'        from comfy.utils import ProgressBar\n        pickface_bar = ProgressBar(max(1, end_frame - start_frame + 1))\n        masks_dict = {}\'\'\'),\n        (\'                # Periodic cleanup and memory monitoring\', \'\'\'                pickface_bar.update(1)\n                # Periodic cleanup and memory monitoring\'\'\'),\n        (\'        for frame_idx in range(num_frames):\', \'\'\'        from comfy.utils import ProgressBar\n        pickface_bar = ProgressBar(max(1, num_frames))\n        for frame_idx in range(num_frames):\'\'\'),\n        (\'            # Flush to disk periodically and free memory\', \'\'\'            pickface_bar.update(1)\n            # Flush to disk periodically and free memory\'\'\')]\n    for old,new in replacements:\n        assert source.count(old) == 1, old\n        source = source.replace(old,new)\n    source += \'\\n\' + MARKER + \'\\n\'\n    ast.parse(source)\n    return source\n\n\ndef prepare(root):\n    p = Path(root) / \'ComfyUI/custom_nodes/comfyui-sam3/nodes/sam3_video_nodes.py\'\n    source = p.read_text()\n    new = patch(source)\n    if new != source:\n        p.with_suffix(\'.py.pre-v5_7\').write_text(source)\n        p.write_text(new)\n')
progress_spec = importlib.util.spec_from_file_location('pickface_progress_patch', ROOT / 'progress-patch-v5_7.py')
progress_patch = importlib.util.module_from_spec(progress_spec)
progress_spec.loader.exec_module(progress_patch)
COMFY = ROOT / 'ComfyUI'
LOG = ROOT / 'comfyui.log'
if 'COMFY_PROCESS' in globals() and COMFY_PROCESS.poll() is None:
    if not globals().get('V5_7_SERVER_STARTED', False):
        raise RuntimeError('An older server is running. Use a fresh Colab runtime for v5.')
    print('PickFace server is already running, PID:', COMFY_PROCESS.pid)
else:
    with socket.socket() as probe:
        if probe.connect_ex(('127.0.0.1', PORT)) == 0:
            raise RuntimeError('Port is already occupied. Use a fresh runtime; existing processes are left untouched.')
    window_patch.restore_previous_patch(ROOT)
    pf4.prepare(ROOT)
    window_patch.prepare(ROOT)
    sam_patch.prepare(ROOT)
    progress_patch.prepare(ROOT)
    SERVER_ENV = RUN_ENV.copy()
    SERVER_ENV['IT_DROP_FRAMES'] = (ROOT / 'payload/settings/it_drop_frames.txt').read_text().strip()
    LOG_HANDLE = LOG.open('ab')
    COMFY_PROCESS = subprocess.Popen(
        [str(PYTHON), 'main.py', '--listen', '127.0.0.1', '--port', str(PORT), '--disable-auto-launch'],
        cwd=COMFY, env=SERVER_ENV, stdout=LOG_HANDLE, stderr=subprocess.STDOUT)
    V5_7_SERVER_STARTED = True
    print('Starting the generation server...', flush=True)
for attempt in range(180):
    if COMFY_PROCESS.poll() is not None:
        print(LOG.read_text(errors='replace')[-18000:])
        raise RuntimeError('Server stopped. See the log above.')
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/object_info/PickFaceSelectionInfo', timeout=3) as response:
            assert 'PickFaceSelectionInfo' in json.load(response)
        break
    except (urllib.error.URLError, TimeoutError, AssertionError):
        if attempt % 10 == 0:
            print(f'Waiting for server: {attempt * 2} seconds', flush=True)
        time.sleep(2)
else:
    print(LOG.read_text(errors='replace')[-18000:])
    raise RuntimeError('Server startup timed out. Review the log; process has been left running.')
run_visible([str(PYTHON), '-u', str(HELPER), 'check', '--root', str(ROOT), '--port', str(PORT)],
            'Checking nodes and models', RUN_ENV)
print('Ready. Continue to step 7. No ComfyUI browser tab is needed.')

(ROOT / 'controller-v5_3.py').write_text('"""Colab-side API client. No browser tab or public tunnel is required."""\nimport copy\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\nfrom urllib.request import Request, urlopen\nfrom urllib.error import HTTPError\nfrom urllib.parse import urlencode\n\n\ndef ancestors(graph, targets):\n    keep = set()\n    def visit(node):\n        if node in keep:\n            return\n        keep.add(node)\n        for value in graph[node][\'inputs\'].values():\n            if isinstance(value, list) and len(value) == 2 and str(value[0]) in graph:\n                visit(str(value[0]))\n    for node in targets:\n        visit(node)\n    return {key: value for key, value in graph.items() if key in keep}\n\n\ndef make_prompt(template, video, audio, face_id, mode, resolution="720p"):\n    if resolution not in ("480p", "720p"):\n        raise ValueError("Select 480p or 720p")\n    graph = copy.deepcopy(template)\n    width, height = (1280, 720) if resolution == "720p" else (832, 480)\n    graph["245"]["inputs"]["value"] = width\n    graph["246"]["inputs"]["value"] = height\n    graph[\'373\'][\'inputs\'][\'video\'] = video\n    graph[\'377\'][\'inputs\'][\'audio\'] = audio\n    graph[\'374\'][\'inputs\'].update(base_precision=\'bf16\', attention_mode=\'sdpa\')\n    graph[\'600\'] = {\'class_type\': \'PickFaceSelectionInfo\', \'inputs\': {\n        \'masks\': [\'502\', 0], \'selected_id\': -1 if mode == \'detect\' else int(face_id)}}\n    graph[\'503\'][\'inputs\'].update(masks=[\'600\', 0],\n        obj_id=-1 if mode == \'detect\' else int(face_id), plot_all_masks=mode == \'detect\')\n    if mode not in (\'detect\', \'preview\', \'generate\'):\n        raise ValueError(\'Unknown operation\')\n    return ancestors(graph, [\'131\'] if mode == \'generate\' else [\'510\'])\n\n\nclass Client:\n    def __init__(self, root, port=8188):\n        self.root = Path(root)\n        self.base = f\'http://127.0.0.1:{port}\'\n        self.state_path = self.root / \'colab-v5-state.json\'\n        self.state = json.loads(self.state_path.read_text()) if self.state_path.exists() else {}\n\n    def api(self, path, data=None):\n        body = None if data is None else json.dumps(data).encode()\n        req = Request(self.base + path, data=body, headers={\'Content-Type\': \'application/json\'})\n        try:\n            with urlopen(req, timeout=30) as response:\n                return json.load(response)\n        except HTTPError as exc:\n            raise RuntimeError(f\'ComfyUI HTTP {exc.code}: {exc.read().decode()[:5000]}\') from exc\n\n    def save(self):\n        self.state_path.write_text(json.dumps(self.state, indent=2))\n\n    def submit(self, graph, mode, context):\n        q = self.api(\'/queue\')\n        if q.get(\'queue_running\') or q.get(\'queue_pending\'):\n            raise RuntimeError(\'ComfyUI is busy. Wait for the current job, then use Recover last job.\')\n        # Persist the exact submitted graph for diagnostics; no UI workflow conversion.\n        (self.root / \'last-prompt-v5.json\').write_text(json.dumps(graph, indent=2))\n        result = self.api(\'/prompt\', {\'prompt\': graph, \'client_id\': str(uuid.uuid4())})\n        if result.get(\'node_errors\') or \'prompt_id\' not in result:\n            raise RuntimeError(json.dumps(result))\n        self.state = {\'prompt_id\': result[\'prompt_id\'], \'mode\': mode, \'context\': context}\n        self.save()\n        return self.state[\'prompt_id\']\n\n    def wait(self, report=print):\n        prompt_id = self.state.get(\'prompt_id\')\n        if not prompt_id:\n            raise RuntimeError(\'No submitted job to recover.\')\n        started = time.monotonic()\n        while True:\n            entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n            if entry:\n                status = entry.get(\'status\', {})\n                if status.get(\'status_str\') == \'error\':\n                    details = [v.get(\'exception_message\', kind) for kind, v in status.get(\'messages\', [])\n                               if kind in (\'execution_error\', \'execution_interrupted\')]\n                    raise RuntimeError(\'\\n\'.join(details) or \'Job failed. Download diagnostics in step 9.\')\n                if status.get(\'completed\'):\n                    return entry\n            q = self.api(\'/queue\')\n            jobs = q.get(\'queue_running\', []) + q.get(\'queue_pending\', [])\n            if not entry and not any(item[1] == prompt_id for item in jobs):\n                # History can become available between the two requests.\n                entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n                if not entry:\n                    raise RuntimeError(\'Job is no longer in the queue or history. Detect faces again.\')\n            report(f\'{self.state["mode"].capitalize()} in progress — {int(time.monotonic()-started)} seconds\')\n            time.sleep(3)\n\n    def image(self, item):\n        query = urlencode({k: item[k] for k in (\'filename\', \'subfolder\', \'type\')})\n        with urlopen(self.base + \'/view?\' + query, timeout=30) as response:\n            return response.read()\n')
(ROOT / 'import-files-v5_3.py').write_text('"""Import media already uploaded through Colab\'s Files sidebar."""\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport uuid\n\n\ndef check_file(value, kind):\n    value = value.strip().strip(\'"\').strip("\'")\n    if not value:\n        raise ValueError(f\'Paste the {kind} file path from the Files sidebar.\')\n    path = Path(value).expanduser().resolve()\n    if not path.is_file():\n        raise ValueError(f\'File not found: {path}. Finish uploading it in Files first.\')\n    allowed = {\'.mp4\', \'.mov\', \'.mkv\', \'.webm\', \'.avi\'} if kind == \'video\' else {\'.wav\', \'.mp3\', \'.flac\', \'.m4a\', \'.ogg\', \'.aac\'}\n    if path.suffix.lower() not in allowed:\n        raise ValueError(f\'Unsupported {kind} file extension: {path.suffix}\')\n    result = subprocess.run([\'ffprobe\', \'-v\', \'error\', \'-show_entries\', \'stream=codec_type\',\n                             \'-of\', \'json\', str(path)], capture_output=True, text=True, timeout=60)\n    if result.returncode:\n        raise ValueError(f\'Cannot read {kind} file. Check that its upload has finished.\')\n    streams = json.loads(result.stdout).get(\'streams\', [])\n    if not any(s.get(\'codec_type\') == kind for s in streams):\n        raise ValueError(f\'The selected file has no {kind} stream.\')\n    return path\n\n\ndef import_pair(video, audio, destination):\n    # Validate both files before copying either one. Never alter the originals.\n    sources = [check_file(video, \'video\'), check_file(audio, \'audio\')]\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    imported = []\n    for kind, source in zip((\'video\', \'audio\'), sources):\n        if source.parent == destination:\n            target = source\n        else:\n            target = destination / f\'colab_{kind}_{uuid.uuid4().hex[:12]}{source.suffix.lower()}\'\n            temporary = target.with_suffix(target.suffix + \'.partial\')\n            try:\n                shutil.copyfile(source, temporary)\n                temporary.replace(target)\n            finally:\n                if temporary.exists():\n                    temporary.unlink()\n        imported.append((source.name, target.name))\n    return imported\n')

(ROOT / 'controller-v5_3.py').write_text('"""Colab-side API client. No browser tab or public tunnel is required."""\nimport copy\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\nfrom urllib.request import Request, urlopen\nfrom urllib.error import HTTPError\nfrom urllib.parse import urlencode\n\n\ndef ancestors(graph, targets):\n    keep = set()\n    def visit(node):\n        if node in keep:\n            return\n        keep.add(node)\n        for value in graph[node][\'inputs\'].values():\n            if isinstance(value, list) and len(value) == 2 and str(value[0]) in graph:\n                visit(str(value[0]))\n    for node in targets:\n        visit(node)\n    return {key: value for key, value in graph.items() if key in keep}\n\n\ndef make_prompt(template, video, audio, face_id, mode, resolution="720p"):\n    if resolution not in ("480p", "720p"):\n        raise ValueError("Select 480p or 720p")\n    graph = copy.deepcopy(template)\n    width, height = (1280, 720) if resolution == "720p" else (832, 480)\n    graph["245"]["inputs"]["value"] = width\n    graph["246"]["inputs"]["value"] = height\n    graph[\'373\'][\'inputs\'][\'video\'] = video\n    graph[\'373\'][\'inputs\'][\'force_rate\'] = 25.0\n    graph[\'270\'][\'inputs\'][\'value\'] = 250\n    graph[\'377\'][\'inputs\'][\'duration\'] = 10.0\n    graph[\'377\'][\'inputs\'][\'audio\'] = audio\n    graph[\'374\'][\'inputs\'].update(base_precision=\'bf16\', attention_mode=\'sdpa\')\n    graph[\'600\'] = {\'class_type\': \'PickFaceSelectionInfo\', \'inputs\': {\n        \'masks\': [\'502\', 0], \'selected_id\': -1 if mode == \'detect\' else int(face_id)}}\n    graph[\'503\'][\'inputs\'].update(masks=[\'600\', 0],\n        obj_id=-1 if mode == \'detect\' else int(face_id), plot_all_masks=mode == \'detect\')\n    if mode not in (\'detect\', \'preview\', \'generate\'):\n        raise ValueError(\'Unknown operation\')\n    return ancestors(graph, [\'131\'] if mode == \'generate\' else [\'510\'])\n\n\nclass Client:\n    def __init__(self, root, port=8188):\n        self.root = Path(root)\n        self.base = f\'http://127.0.0.1:{port}\'\n        self.state_path = self.root / \'colab-v5-state.json\'\n        self.state = json.loads(self.state_path.read_text()) if self.state_path.exists() else {}\n\n    def api(self, path, data=None):\n        body = None if data is None else json.dumps(data).encode()\n        req = Request(self.base + path, data=body, headers={\'Content-Type\': \'application/json\'})\n        try:\n            with urlopen(req, timeout=30) as response:\n                return json.load(response)\n        except HTTPError as exc:\n            raise RuntimeError(f\'ComfyUI HTTP {exc.code}: {exc.read().decode()[:5000]}\') from exc\n\n    def save(self):\n        self.state_path.write_text(json.dumps(self.state, indent=2))\n\n    def submit(self, graph, mode, context):\n        q = self.api(\'/queue\')\n        if q.get(\'queue_running\') or q.get(\'queue_pending\'):\n            raise RuntimeError(\'ComfyUI is busy. Wait for the current job, then use Recover last job.\')\n        # Persist the exact submitted graph for diagnostics; no UI workflow conversion.\n        (self.root / \'last-prompt-v5.json\').write_text(json.dumps(graph, indent=2))\n        result = self.api(\'/prompt\', {\'prompt\': graph, \'client_id\': str(uuid.uuid4())})\n        if result.get(\'node_errors\') or \'prompt_id\' not in result:\n            raise RuntimeError(json.dumps(result))\n        self.state = {\'prompt_id\': result[\'prompt_id\'], \'mode\': mode, \'context\': context}\n        self.save()\n        return self.state[\'prompt_id\']\n\n    def wait(self, report=print):\n        prompt_id = self.state.get(\'prompt_id\')\n        if not prompt_id:\n            raise RuntimeError(\'No submitted job to recover.\')\n        started = time.monotonic()\n        while True:\n            entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n            if entry:\n                status = entry.get(\'status\', {})\n                if status.get(\'status_str\') == \'error\':\n                    details = [v.get(\'exception_message\', kind) for kind, v in status.get(\'messages\', [])\n                               if kind in (\'execution_error\', \'execution_interrupted\')]\n                    raise RuntimeError(\'\\n\'.join(details) or \'Job failed. Download diagnostics in step 9.\')\n                if status.get(\'completed\'):\n                    return entry\n            q = self.api(\'/queue\')\n            jobs = q.get(\'queue_running\', []) + q.get(\'queue_pending\', [])\n            if not entry and not any(item[1] == prompt_id for item in jobs):\n                # History can become available between the two requests.\n                entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n                if not entry:\n                    raise RuntimeError(\'Job is no longer in the queue or history. Detect faces again.\')\n            report(f\'{self.state["mode"].capitalize()} in progress — {int(time.monotonic()-started)} seconds\')\n            time.sleep(3)\n\n    def image(self, item):\n        query = urlencode({k: item[k] for k in (\'filename\', \'subfolder\', \'type\')})\n        with urlopen(self.base + \'/view?\' + query, timeout=30) as response:\n            return response.read()\n')

(ROOT / 'controller-v5_5.py').write_text('"""Colab-side API client. No browser tab or public tunnel is required."""\nimport copy\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\nfrom urllib.request import Request, urlopen\nfrom urllib.error import HTTPError\nfrom urllib.parse import urlencode\n\n\ndef ancestors(graph, targets):\n    keep = set()\n    def visit(node):\n        if node in keep:\n            return\n        keep.add(node)\n        for value in graph[node][\'inputs\'].values():\n            if isinstance(value, list) and len(value) == 2 and str(value[0]) in graph:\n                visit(str(value[0]))\n    for node in targets:\n        visit(node)\n    return {key: value for key, value in graph.items() if key in keep}\n\n\ndef make_prompt(template, video, audio, face_id, mode, resolution="720p", dimensions=None):\n    if resolution not in ("480p", "720p"):\n        raise ValueError("Select 480p or 720p")\n    graph = copy.deepcopy(template)\n    if dimensions is None or len(dimensions) != 2:\n        raise ValueError("Import a video in step 7 to determine its dimensions")\n    width, height = dimensions\n    if any(not isinstance(v, int) or v < 16 or v % 16 for v in dimensions):\n        raise ValueError("Video dimensions must be positive multiples of 16")\n    graph["245"]["inputs"]["value"] = width\n    graph["246"]["inputs"]["value"] = height\n    graph[\'373\'][\'inputs\'][\'video\'] = video\n    graph[\'373\'][\'inputs\'][\'force_rate\'] = 25.0\n    graph[\'270\'][\'inputs\'][\'value\'] = 375\n    graph[\'377\'][\'inputs\'][\'duration\'] = 15.0\n    graph[\'377\'][\'inputs\'][\'audio\'] = audio\n    graph[\'374\'][\'inputs\'].update(base_precision=\'bf16\', attention_mode=\'sdpa\')\n    graph[\'600\'] = {\'class_type\': \'PickFaceSelectionInfo\', \'inputs\': {\n        \'masks\': [\'502\', 0], \'selected_id\': -1 if mode == \'detect\' else int(face_id)}}\n    graph[\'503\'][\'inputs\'].update(masks=[\'600\', 0],\n        obj_id=-1 if mode == \'detect\' else int(face_id), plot_all_masks=mode == \'detect\')\n    if mode not in (\'detect\', \'preview\', \'generate\'):\n        raise ValueError(\'Unknown operation\')\n    return ancestors(graph, [\'131\'] if mode == \'generate\' else [\'510\'])\n\n\nclass Client:\n    def __init__(self, root, port=8188):\n        self.root = Path(root)\n        self.base = f\'http://127.0.0.1:{port}\'\n        self.state_path = self.root / \'colab-v5-state.json\'\n        self.state = json.loads(self.state_path.read_text()) if self.state_path.exists() else {}\n\n    def api(self, path, data=None):\n        body = None if data is None else json.dumps(data).encode()\n        req = Request(self.base + path, data=body, headers={\'Content-Type\': \'application/json\'})\n        try:\n            with urlopen(req, timeout=30) as response:\n                return json.load(response)\n        except HTTPError as exc:\n            raise RuntimeError(f\'ComfyUI HTTP {exc.code}: {exc.read().decode()[:5000]}\') from exc\n\n    def save(self):\n        self.state_path.write_text(json.dumps(self.state, indent=2))\n\n    def submit(self, graph, mode, context):\n        q = self.api(\'/queue\')\n        if q.get(\'queue_running\') or q.get(\'queue_pending\'):\n            raise RuntimeError(\'ComfyUI is busy. Wait for the current job, then use Recover last job.\')\n        # Persist the exact submitted graph for diagnostics; no UI workflow conversion.\n        (self.root / \'last-prompt-v5.json\').write_text(json.dumps(graph, indent=2))\n        result = self.api(\'/prompt\', {\'prompt\': graph, \'client_id\': str(uuid.uuid4())})\n        if result.get(\'node_errors\') or \'prompt_id\' not in result:\n            raise RuntimeError(json.dumps(result))\n        self.state = {\'prompt_id\': result[\'prompt_id\'], \'mode\': mode, \'context\': context}\n        self.save()\n        return self.state[\'prompt_id\']\n\n    def wait(self, report=print):\n        prompt_id = self.state.get(\'prompt_id\')\n        if not prompt_id:\n            raise RuntimeError(\'No submitted job to recover.\')\n        started = time.monotonic()\n        while True:\n            entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n            if entry:\n                status = entry.get(\'status\', {})\n                if status.get(\'status_str\') == \'error\':\n                    details = [v.get(\'exception_message\', kind) for kind, v in status.get(\'messages\', [])\n                               if kind in (\'execution_error\', \'execution_interrupted\')]\n                    raise RuntimeError(\'\\n\'.join(details) or \'Job failed. Download diagnostics in step 9.\')\n                if status.get(\'completed\'):\n                    return entry\n            q = self.api(\'/queue\')\n            jobs = q.get(\'queue_running\', []) + q.get(\'queue_pending\', [])\n            if not entry and not any(item[1] == prompt_id for item in jobs):\n                # History can become available between the two requests.\n                entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n                if not entry:\n                    raise RuntimeError(\'Job is no longer in the queue or history. Detect faces again.\')\n            report(f\'{self.state["mode"].capitalize()} in progress — {int(time.monotonic()-started)} seconds\')\n            time.sleep(3)\n\n    def image(self, item):\n        query = urlencode({k: item[k] for k in (\'filename\', \'subfolder\', \'type\')})\n        with urlopen(self.base + \'/view?\' + query, timeout=30) as response:\n            return response.read()\n')

(ROOT / 'geometry-v5_5.py').write_text('"""Read dimensions with the same decoder used by VHS, including rotation tags."""\nimport json\nimport subprocess\n\nLONG_SIDES = {\'720p\': 1280, \'480p\': 832}\n\n\ndef output_size(width, height, resolution):\n    if resolution not in LONG_SIDES:\n        raise ValueError(\'Select 480p or 720p\')\n    if width <= 0 or height <= 0:\n        raise ValueError(\'Invalid source video dimensions\')\n    longest = LONG_SIDES[resolution]\n    scale = longest / max(width, height)\n    return tuple(max(16, int(side * scale / 16 + 0.5) * 16) for side in (width, height))\n\n\ndef probe_video(path, python):\n    script = \'\'\'import cv2, json, sys\ncap = cv2.VideoCapture(sys.argv[1])\ntry:\n    ok, frame = cap.read()\n    if not ok:\n        raise ValueError(\'Could not decode the first video frame\')\n    print(json.dumps([int(frame.shape[1]), int(frame.shape[0])]))\nfinally:\n    cap.release()\n\'\'\'\n    result = subprocess.run([str(python), \'-c\', script, str(path)],\n                            capture_output=True, text=True, timeout=60)\n    if result.returncode:\n        raise ValueError(\'Cannot decode video: \' + result.stderr[-1000:])\n    width, height = json.loads(result.stdout)\n    if width <= 0 or height <= 0:\n        raise ValueError(\'Invalid video dimensions\')\n    return width, height\n')

(ROOT / 'controller-v5_7.py').write_text('"""Colab-side API client. No browser tab or public tunnel is required."""\nfrom pickface_progress_client import Progress\nimport copy\nimport json\nimport time\nimport uuid\nfrom pathlib import Path\nfrom urllib.request import Request, urlopen\nfrom urllib.error import HTTPError\nfrom urllib.parse import urlencode\n\n\ndef ancestors(graph, targets):\n    keep = set()\n    def visit(node):\n        if node in keep:\n            return\n        keep.add(node)\n        for value in graph[node][\'inputs\'].values():\n            if isinstance(value, list) and len(value) == 2 and str(value[0]) in graph:\n                visit(str(value[0]))\n    for node in targets:\n        visit(node)\n    return {key: value for key, value in graph.items() if key in keep}\n\n\ndef make_prompt(template, video, audio, face_id, mode, resolution="720p", dimensions=None):\n    if resolution not in ("480p", "720p"):\n        raise ValueError("Select 480p or 720p")\n    graph = copy.deepcopy(template)\n    if dimensions is None or len(dimensions) != 2:\n        raise ValueError("Import a video in step 7 to determine its dimensions")\n    width, height = dimensions\n    if any(not isinstance(v, int) or v < 16 or v % 16 for v in dimensions):\n        raise ValueError("Video dimensions must be positive multiples of 16")\n    graph["245"]["inputs"]["value"] = width\n    graph["246"]["inputs"]["value"] = height\n    graph[\'373\'][\'inputs\'][\'video\'] = video\n    graph[\'373\'][\'inputs\'][\'force_rate\'] = 25.0\n    graph[\'270\'][\'inputs\'][\'value\'] = 500\n    graph[\'377\'][\'inputs\'][\'duration\'] = 20.0\n    graph[\'377\'][\'inputs\'][\'audio\'] = audio\n    graph[\'194\'][\'inputs\'].update(audio_scale=1.2, audio_cfg_scale=1.2)\n    graph[\'374\'][\'inputs\'].update(base_precision=\'bf16\', attention_mode=\'sdpa\')\n    graph[\'600\'] = {\'class_type\': \'PickFaceSelectionInfo\', \'inputs\': {\n        \'masks\': [\'502\', 0], \'selected_id\': -1 if mode == \'detect\' else int(face_id)}}\n    graph[\'503\'][\'inputs\'].update(masks=[\'600\', 0],\n        obj_id=-1 if mode == \'detect\' else int(face_id), plot_all_masks=mode == \'detect\')\n    if mode not in (\'detect\', \'preview\', \'generate\'):\n        raise ValueError(\'Unknown operation\')\n    return ancestors(graph, [\'131\'] if mode == \'generate\' else [\'510\'])\n\n\nclass Client:\n    def __init__(self, root, port=8188):\n        self.root = Path(root)\n        self.base = f\'http://127.0.0.1:{port}\'\n        self.state_path = self.root / \'colab-v5-state.json\'\n        self.state = json.loads(self.state_path.read_text()) if self.state_path.exists() else {}\n\n    def api(self, path, data=None):\n        body = None if data is None else json.dumps(data).encode()\n        req = Request(self.base + path, data=body, headers={\'Content-Type\': \'application/json\'})\n        try:\n            with urlopen(req, timeout=30) as response:\n                return json.load(response)\n        except HTTPError as exc:\n            raise RuntimeError(f\'ComfyUI HTTP {exc.code}: {exc.read().decode()[:5000]}\') from exc\n\n    def save(self):\n        self.state_path.write_text(json.dumps(self.state, indent=2))\n\n    def submit(self, graph, mode, context):\n        q = self.api(\'/queue\')\n        if q.get(\'queue_running\') or q.get(\'queue_pending\'):\n            raise RuntimeError(\'ComfyUI is busy. Wait for the current job, then use Recover last job.\')\n        # Persist the exact submitted graph for diagnostics; no UI workflow conversion.\n        (self.root / \'last-prompt-v5.json\').write_text(json.dumps(graph, indent=2))\n        result = self.api(\'/prompt\', {\'prompt\': graph, \'client_id\': str(uuid.uuid4())})\n        if result.get(\'node_errors\') or \'prompt_id\' not in result:\n            raise RuntimeError(json.dumps(result))\n        self.state = {\'prompt_id\': result[\'prompt_id\'], \'mode\': mode, \'context\': context}\n        self.save()\n        return self.state[\'prompt_id\']\n\n    def wait(self, report=print, progress=None):\n        prompt_id = self.state.get(\'prompt_id\')\n        if not prompt_id:\n            raise RuntimeError(\'No submitted job to recover.\')\n        graph_path = self.root / \'last-prompt-v5.json\'\n        graph = json.loads(graph_path.read_text()) if graph_path.exists() else {}\n        tracker = Progress(graph)\n        started = time.monotonic()\n        while True:\n            entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n            if entry:\n                status = entry.get(\'status\', {})\n                if status.get(\'status_str\') == \'error\':\n                    details = [v.get(\'exception_message\', kind) for kind, v in status.get(\'messages\', [])\n                               if kind in (\'execution_error\', \'execution_interrupted\')]\n                    raise RuntimeError(\'\\n\'.join(details) or \'Job failed. Download diagnostics in step 9.\')\n                if status.get(\'completed\'):\n                    if progress:\n                        progress({\'percent\':100, \'stage_percent\':100, \'label\':\'Completed\'})\n                    return entry\n            q = self.api(\'/queue\')\n            jobs = q.get(\'queue_running\', []) + q.get(\'queue_pending\', [])\n            if not entry and not any(item[1] == prompt_id for item in jobs):\n                # History can become available between the two requests.\n                entry = self.api(\'/history/\' + prompt_id).get(prompt_id)\n                if not entry:\n                    raise RuntimeError(\'Job is no longer in the queue or history. Detect faces again.\')\n            if progress:\n                try:\n                    snapshot = self.api(\'/pickface/progress/\' + prompt_id)\n                    if snapshot.get(\'prompt_id\') == prompt_id:\n                        progress(tracker.snapshot(snapshot.get(\'nodes\', {})))\n                except (OSError, RuntimeError, ValueError):\n                    progress({\'percent\':tracker.percent, \'stage_percent\':0,\n                              \'label\':\'Progress temporarily unavailable; checking completion\'})\n            report(f\'{self.state["mode"].capitalize()} in progress — {int(time.monotonic()-started)} seconds\')\n            time.sleep(3)\n\n    def image(self, item):\n        query = urlencode({k: item[k] for k in (\'filename\', \'subfolder\', \'type\')})\n        with urlopen(self.base + \'/view?\' + query, timeout=30) as response:\n            return response.read()\n')

(ROOT / 'pickface_progress_client.py').write_text('"""Monotonic stage-weighted estimate; completion is confirmed by history only."""\nSTAGES = {\n \'VHS_LoadVideo\': (\'Loading video\', 3),\n \'SAM3VideoSegmentation\': (\'Preparing face detection\', 6),\n \'LoadSAM3Model\': (\'Loading face detector\', 3),\n \'SAM3Propagate\': (\'Tracking faces\', 25),\n \'SAM3VideoOutput\': (\'Preparing face preview\', 5),\n \'PreviewImage\': (\'Saving preview images\', 2),\n \'WanVideoModelLoader\': (\'Loading video model\', 5),\n \'WanVideoEncode\': (\'Encoding source video\', 5),\n \'WanVideoSampler\': (\'Generating video\', 75),\n \'WanVideoDecode\': (\'Decoding video\', 5),\n \'VHS_VideoCombine\': (\'Saving video and audio\', 3),\n \'MultiTalkWav2VecEmbeds\': (\'Preparing speech\', 3),\n \'MelBandRoFormerSampler\': (\'Processing audio\', 3),\n}\n\nclass Progress:\n    def __init__(self, graph):\n        self.graph = graph\n        self.percent = 0\n        self.fractions = {}\n\n    def snapshot(self, nodes):\n        done = set()\n        def parents(key):\n            for value in self.graph.get(key, {}).get(\'inputs\', {}).values():\n                if isinstance(value,list) and len(value)==2 and str(value[0]) in self.graph:\n                    parent = str(value[0])\n                    if parent not in done:\n                        done.add(parent); parents(parent)\n        for key, node in nodes.items():\n            if node[\'state\'] in (\'running\',\'finished\'):\n                parents(key)  # Dependencies must have completed or come from cache.\n            if node[\'state\'] == \'finished\': done.add(key)\n        current = None\n        for key in self.graph:\n            node = nodes.get(key, {})\n            fraction = 1.0 if key in done else 0.0\n            if node.get(\'state\') == \'running\' and key not in done:\n                maximum = node.get(\'max\', 0)\n                fraction = min(.99, max(0, node.get(\'value\',0)/maximum)) if maximum else 0\n                current = (key, fraction, node)\n            self.fractions[key] = max(self.fractions.get(key,0), fraction)\n        weight = lambda k: STAGES.get(self.graph[k][\'class_type\'], (\'Preparing resources\',1))[1]\n        total = sum(weight(k) for k in self.graph) or 1\n        value = int(100 * sum(weight(k)*self.fractions[k] for k in self.graph)/total)\n        self.percent = max(self.percent,min(99,value))\n        label, stage_percent = \'Waiting for server\', 0\n        if current:\n            key, fraction, node = current\n            label = STAGES.get(self.graph[key][\'class_type\'], (\'Preparing resources\',1))[0]\n            stage_percent = int(fraction*100)\n            if node.get(\'max\',0) > 1:\n                label += f" ({int(node[\'value\'])}/{int(node[\'max\'])})"\n        return {\'percent\':self.percent,\'stage_percent\':stage_percent,\'label\':label}\n')


In [ ]:
#@title 7. Select files from the Files sidebar and choose resolution
from google.colab import output
from IPython.display import display
import ipywidgets as widgets
import importlib.util
output.enable_custom_widget_manager()
spec = importlib.util.spec_from_file_location('pick_face_files', ROOT / 'import-files-v5_3.py')
pick_face_files = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pick_face_files)
geometry_spec = importlib.util.spec_from_file_location('pick_face_geometry', ROOT / 'geometry-v5_5.py')
geometry = importlib.util.module_from_spec(geometry_spec)
geometry_spec.loader.exec_module(geometry)
VIDEO_GEOMETRY = {}
INPUT = ROOT / 'ComfyUI/input'
INPUT.mkdir(parents=True, exist_ok=True)
VIDEO_PATH = widgets.Text(description='Video path:', placeholder='/content/my_video.mp4',
                          layout=widgets.Layout(width='95%'))
AUDIO_PATH = widgets.Text(description='Audio path:', placeholder='/content/my_speech.mp3',
                          layout=widgets.Layout(width='95%'))
RESOLUTION = widgets.Dropdown(description='Resolution:', options=[
    ('720p — longest side 1280 px', '720p'),
    ('480p — longest side 832 px', '480p')], value='720p',
    layout=widgets.Layout(width='95%'))
APPLY_FILES = widgets.Button(description='Use files', button_style='info')
VIDEO_SELECT = widgets.Dropdown(description='Video:', options=[], disabled=True,
                               layout=widgets.Layout(width='95%'))
AUDIO_SELECT = widgets.Dropdown(description='Audio:', options=[], disabled=True,
                               layout=widgets.Layout(width='95%'))
INPUT_MESSAGE = widgets.Output()

def apply_files(_=None):
    with INPUT_MESSAGE:
        INPUT_MESSAGE.clear_output()
        APPLY_FILES.disabled = True
        try:
            print('Checking and importing files...', flush=True)
            video, audio = pick_face_files.import_pair(VIDEO_PATH.value, AUDIO_PATH.value, INPUT)
            VIDEO_GEOMETRY[video[1]] = geometry.probe_video(INPUT / video[1], PYTHON)
            VIDEO_SELECT.options = [video]
            AUDIO_SELECT.options = [audio]
            show_dimensions()
            print('Ready. Continue to Detect faces in step 8.')
            print('Video:', video[0], '| Audio:', audio[0], '| Resolution:', RESOLUTION.value)
        except Exception as exc:
            # Prevent running with a previous selection after a failed replacement.
            VIDEO_SELECT.options = []
            AUDIO_SELECT.options = []
            print('Could not use these files:', exc)
        finally:
            APPLY_FILES.disabled = False

SIZE_MESSAGE = widgets.HTML()
def show_dimensions(change=None):
    source = VIDEO_GEOMETRY.get(VIDEO_SELECT.value)
    if source:
        width, height = geometry.output_size(*source, RESOLUTION.value)
        SIZE_MESSAGE.value = f'Output: <b>{width} × {height}</b> · up to 20 seconds · 25 FPS'
    else:
        SIZE_MESSAGE.value = ''
RESOLUTION.observe(show_dimensions, names='value')
VIDEO_SELECT.observe(show_dimensions, names='value')
APPLY_FILES.on_click(apply_files)
display(widgets.HTML('<b>Upload your video and audio using the Files sidebar on the left.</b><br>'
    'Wait for each upload to finish. Right-click each file, choose <b>Copy path</b>, '
    'and paste the full path below. Click <b>Use files</b>.<br>'
    'Landscape, portrait and square videos are supported. The longest side follows the selected mode. '
    'Dimensions are rounded to multiples of 16; a few edge pixels may be cropped. '
    'Use video and audio at least 20 seconds long for this test.'),
    VIDEO_PATH, AUDIO_PATH, RESOLUTION, APPLY_FILES, VIDEO_SELECT, AUDIO_SELECT, SIZE_MESSAGE, INPUT_MESSAGE)


In [ ]:
#@title 8. Detect faces, choose a character, and generate
import importlib.util
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image, Video, clear_output
spec = importlib.util.spec_from_file_location('pick_face_client', ROOT / 'controller-v5_7.py')
pf5 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pf5)
CLIENT = pf5.Client(ROOT, PORT)
TEMPLATE = json.loads((ROOT / 'prompt-v5.json').read_text())
DETECT = widgets.Button(description='Detect faces', button_style='info')
FACE = widgets.Dropdown(options=[], description='Face ID:', disabled=True)
CHECK_FACE = widgets.Button(description='Preview selected face', disabled=True,
                            layout=widgets.Layout(width='200px'))
GENERATE = widgets.Button(description='Generate video', button_style='success', disabled=True)
RECOVER = widgets.Button(description='Recover last job')
FRAME = widgets.IntSlider(description='Frame:', min=1, max=1, value=1,
                          continuous_update=False, disabled=True)
FRAME_SELECT = widgets.Dropdown(description='Frame:', options=[1], value=1, disabled=True)
def select_frame(change):
    FRAME.value = change['new']
FRAME_SELECT.observe(select_frame, names='value')
STATUS = widgets.HTML()
PROGRESS = widgets.IntProgress(value=0, min=0, max=100, description='Overall:',
    layout=widgets.Layout(width='95%'))
STAGE_PROGRESS = widgets.IntProgress(value=0, min=0, max=100, description='Stage:',
    layout=widgets.Layout(width='95%'))
PROGRESS_TEXT = widgets.HTML('Ready')
def show_progress(data):
    import html
    PROGRESS.value = data['percent']
    STAGE_PROGRESS.value = data['stage_percent']
    PROGRESS_TEXT.value = (f"<b>{data['percent']}%</b> overall · "
        + html.escape(data['label']) + f" · stage {data['stage_percent']}%")
    if data['percent'] == 100:
        PROGRESS.bar_style = STAGE_PROGRESS.bar_style = 'success'

def reset_progress():
    PROGRESS.value = STAGE_PROGRESS.value = 0
    PROGRESS.bar_style = STAGE_PROGRESS.bar_style = 'info'
    PROGRESS_TEXT.value = 'Waiting to start'

FACE_IMAGE = widgets.Output()
VIDEO_OUTPUT = widgets.Output()
SESSION = {'context': None, 'ids': [], 'images': [], 'confirmed': None, 'busy': False}

def input_context():
    source = VIDEO_GEOMETRY.get(VIDEO_SELECT.value)
    dimensions = list(geometry.output_size(*source, RESOLUTION.value)) if source else None
    return {'video': VIDEO_SELECT.value, 'audio': AUDIO_SELECT.value,
            'resolution': RESOLUTION.value, 'dimensions': dimensions, 'max_seconds': 20}

def update_controls():
    busy = SESSION['busy']
    valid = SESSION['context'] == input_context() and bool(SESSION['ids'])
    for button in [DETECT, RECOVER, APPLY_FILES, VIDEO_PATH, AUDIO_PATH]:
        button.disabled = busy
    RESOLUTION.disabled = busy
    FACE.disabled = busy or not valid
    CHECK_FACE.disabled = busy or not valid
    GENERATE.disabled = busy or not valid or SESSION['confirmed'] != FACE.value

def invalidate(change=None):
    reset_progress()
    SESSION.update(context=None, ids=[], images=[], confirmed=None)
    FACE.options = []
    FRAME.disabled = True
    FRAME_SELECT.disabled = True
    with FACE_IMAGE:
        clear_output()
    STATUS.value = 'Inputs changed. Click <b>Detect faces</b>.'
    update_controls()

def face_changed(change):
    SESSION['confirmed'] = None
    update_controls()

def show_frame(change=None):
    with FACE_IMAGE:
        clear_output(wait=True)
        if SESSION['images']:
            try:
                display(Image(data=CLIENT.image(SESSION['images'][FRAME.value - 1]), width=832))
            except Exception as exc:
                print('Could not load preview:', exc)

def present(entry):
    context = CLIENT.state['context']
    if context != input_context():
        raise RuntimeError('This job used different inputs. Select those inputs or detect faces again.')
    mode = CLIENT.state['mode']
    if mode in ('detect', 'preview'):
        data = json.loads(entry['outputs']['600']['text'][0])
        SESSION['ids'] = data['face_ids']
        SESSION['visible_frames'] = data.get('visible_frames', {})
        SESSION['context'] = context
        previous = FACE.value
        FACE.options = [(f'Face {i}', i) for i in SESSION['ids']]
        if previous in SESSION['ids']:
            FACE.value = previous
        SESSION['images'] = entry['outputs']['510']['images']
        FRAME.max = max(1, len(SESSION['images']))
        # SAM propagation starts at zero-based frame 1 in the original graph.
        # Its first output image has no overlay; open the first propagated frame.
        first_labeled = min(FRAME.max, max(1, int(TEMPLATE['502']['inputs']['start_frame']) + 1))
        FRAME_SELECT.options = list(range(1, FRAME.max + 1))
        FRAME_SELECT.value = first_labeled
        FRAME.value = first_labeled
        FRAME.disabled = not bool(SESSION['images'])
        FRAME_SELECT.disabled = FRAME.disabled
        show_frame()
        if mode == 'detect':
            SESSION['confirmed'] = None
            STATUS.value = ('Match a number on the image to <b>Face ID</b>, then click '
                            '<b>Preview selected face</b>. IDs are not left-to-right positions.')
        else:
            selected = CLIENT.state.get('face_id')
            SESSION['confirmed'] = selected if selected == FACE.value else None
            STATUS.value = 'Check the highlighted face. If it is correct, click <b>Generate video</b>.'
        if not SESSION['ids']:
            STATUS.value = ('No faces were detected. Try a clip '
                            'where the intended face is clearly visible.')
    else:
        outputs = entry.get('outputs', {}).get('131', {}).get('gifs', [])
        mp4s = [v for v in outputs if v.get('type') == 'output' and v['filename'].endswith('.mp4')]
        if not mp4s:
            raise RuntimeError('No output MP4 was recorded. Download diagnostics in step 9.')
        result = ROOT / 'ComfyUI/output' / mp4s[-1].get('subfolder', '') / mp4s[-1]['filename']
        with VIDEO_OUTPUT:
            clear_output()
            display(Video(str(result), embed=True, width=832))
        STATUS.value = 'Video ready. Use step 9 to download it.'
        if 'refresh_results' in globals():
            refresh_results()

def operate(mode):
    if SESSION['busy']:
        return
    reset_progress()
    SESSION['busy'] = True
    update_controls()
    try:
        context = input_context()
        if mode != 'recover':
            if not context['video'] or (mode == 'generate' and not context['audio']):
                raise RuntimeError('Select a video and, for generation, a speech audio file in step 7.')
            if mode in ('preview', 'generate') and (
                    SESSION['context'] != context or FACE.value not in SESSION['ids']):
                raise RuntimeError('Detect faces for the current inputs first.')
            if mode == 'generate' and SESSION['confirmed'] != FACE.value:
                raise RuntimeError('Preview the selected face before generating.')
            if mode == 'detect':
                SESSION['confirmed'] = None
            graph = pf5.make_prompt(TEMPLATE, context['video'], context['audio'], FACE.value, mode, resolution=RESOLUTION.value, dimensions=context['dimensions'])
            CLIENT.submit(graph, mode, context)
            CLIENT.state['face_id'] = FACE.value
            CLIENT.save()
        entry = CLIENT.wait(lambda text: None, progress=show_progress)
        present(entry)
    except KeyboardInterrupt:
        PROGRESS.bar_style = STAGE_PROGRESS.bar_style = 'warning'
        PROGRESS_TEXT.value = 'Waiting paused. Use Recover last job.'
        STATUS.value = 'Waiting interrupted. The server job may still be running; use Recover last job.'
    except Exception as exc:
        import html
        PROGRESS.bar_style = STAGE_PROGRESS.bar_style = 'danger'
        PROGRESS_TEXT.value = 'Operation failed. See the error below.'
        STATUS.value = '<b>Error:</b> ' + html.escape(str(exc))
    finally:
        SESSION['busy'] = False
        update_controls()

RESOLUTION.observe(invalidate, names='value')
VIDEO_SELECT.observe(invalidate, names='value')
AUDIO_SELECT.observe(invalidate, names='value')
FACE.observe(face_changed, names='value')
FRAME.observe(show_frame, names='value')
DETECT.on_click(lambda _: operate('detect'))
CHECK_FACE.on_click(lambda _: operate('preview'))
GENERATE.on_click(lambda _: operate('generate'))
RECOVER.on_click(lambda _: operate('recover'))
display(widgets.HTML('<b>1.</b> Detect faces. <b>2.</b> Choose the number on the intended face. '
                     '<b>3.</b> Preview that face. <b>4.</b> Generate. '
                     'Use the Frame menu to inspect tracking across the clip. '
                     'A face can be selected even if it disappears briefly; missing masks stay empty.'),
        widgets.HBox([DETECT, FACE, CHECK_FACE]), FRAME_SELECT, FACE_IMAGE,
        widgets.HBox([GENERATE, RECOVER]), PROGRESS, STAGE_PROGRESS, PROGRESS_TEXT,
        widgets.HTML('Overall percentage is estimated by processing stages, not elapsed time. '
                     'It can pause during model loading or a long generation step. '
                     '100% means the server finished successfully.'), STATUS, VIDEO_OUTPUT)


In [ ]:
#@title 9. View and download your results
from google.colab import files
from IPython.display import display, Video
import ipywidgets as widgets
import urllib.request

RESULT = widgets.Dropdown(description='Result:', layout=widgets.Layout(width='95%'))
REFRESH = widgets.Button(description='Refresh results')
PREVIEW = widgets.Button(description='Play video')
DOWNLOAD = widgets.Button(description='Download video')
DIAGNOSTICS = widgets.Button(description='Download diagnostics', layout=widgets.Layout(width='210px'))
RESULT_MESSAGE = widgets.Output()

def refresh_results(_=None):
    with RESULT_MESSAGE:
        RESULT_MESSAGE.clear_output()
        out = ROOT / 'ComfyUI/output'
        candidates = sorted(out.rglob('*.mp4'), key=lambda p:p.stat().st_mtime, reverse=True)
        # Only list completed outputs recorded in ComfyUI history, not in-progress files.
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/history', timeout=20) as r:
            history = json.load(r)
        completed = set()
        for item in history.values():
            if not item.get('status', {}).get('completed'):
                continue
            for node in item.get('outputs', {}).values():
                for values in node.values():
                    if not isinstance(values, list):
                        continue
                    for value in values:
                        if isinstance(value, dict) and value.get('type') == 'output' and value.get('filename'):
                            completed.add((out / value.get('subfolder', '') / value['filename']).resolve())
        candidates = [p for p in candidates if p.resolve() in completed]
        RESULT.options = [(str(p.relative_to(out)), str(p)) for p in candidates]
        print(f'Completed videos: {len(candidates)}. Click Refresh results after a new generation.')

def preview_result(_):
    with RESULT_MESSAGE:
        RESULT_MESSAGE.clear_output()
        if RESULT.value:
            display(Video(RESULT.value, embed=True, width=700))
        else:
            print('No completed videos yet. Generate a video in step 8.')

def download_result(_):
    with RESULT_MESSAGE:
        if RESULT.value:
            files.download(RESULT.value)

def diagnostics(_):
    import zipfile
    target = ROOT / 'pick_face_diagnostics.zip'
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name in ['comfyui.log', 'node-check.json', 'selected-models.json', 'v4-diagnostics.json', 'colab-v5-state.json', 'last-prompt-v5.json', 'prompt-v5.json', 'install-v2.log']:
            p = ROOT / name
            if p.exists():
                archive.write(p, name)
        for endpoint in ['history', 'system_stats']:
            try:
                with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/{endpoint}', timeout=20) as r:
                    archive.writestr(endpoint + '.json', r.read())
            except Exception as e:
                archive.writestr(endpoint + '.error.txt', str(e))
        p = ROOT / 'ComfyUI/user/default/workflows/WF_pick_face_Colab_v4.json'
        if p.exists():
            archive.write(p, p.name)
    files.download(str(target))

REFRESH.on_click(refresh_results)
PREVIEW.on_click(preview_result)
DOWNLOAD.on_click(download_result)
DIAGNOSTICS.on_click(diagnostics)
display(RESULT, widgets.HBox([REFRESH, PREVIEW, DOWNLOAD]), DIAGNOSTICS, RESULT_MESSAGE)
refresh_results()
